# ConvNeXt WBC — Clean Pipeline (3-Class)
**Run every cell top-to-bottom. No modifications needed.**

| Step | What happens |
|------|-------------|
| 1 | Install packages + GPU check |
| 2 | Mount Drive, extract zip, load saved splits |
| 3 | Dataset class + weighted sampler |
| 4 | ConvNeXt-Tiny model |
| 5 | Training loop (AMP + early stopping) |
| 6 | 3-Fold cross-validation + OOF predictions |
| 7 | Final model on held-out test set |
| 8 | Results: confusion matrix, F1, accuracy, loss curves |


## Cell 1 — Install packages

In [ ]:
!pip install timm scikit-learn matplotlib seaborn -q
!pip install albumentations -q
!pip install scikit-image -q

In [ ]:
# ── Cell 1 — Install packages ─────────────────────────────────────
!pip install timm scikit-learn matplotlib seaborn -q
!pip install albumentations -q
!pip install scikit-image -q

import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

In [ ]:
import timm
import albumentations
import skimage

print("All packages already available.")

In [ ]:
try:
    import timm
except:
    !pip install timm -q

## Cell 2 — Imports & config

In [ ]:
# ============================================================
# CELL 1 — IMPORTS + CONFIG (KAGGLE VERSION)
# REPLACE YOUR CURRENT CONFIG CELL WITH THIS
# ============================================================

import os, random, warnings, pickle
import numpy as np
import pandas as pd
import cv2
import torch
import timm
import torch.nn as nn

from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score
)

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ============================================================
# SEED
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ============================================================
# DEVICE
# ============================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device : {device}")

if device.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ============================================================
# KAGGLE DATASET PATH
# ============================================================

BASE_PATH = "/kaggle/input/datasets/lasyapriya16/wbc-bench-2026"

print(f"\nBASE_PATH: {BASE_PATH}")

# ============================================================
# CONFIG
# ============================================================

CFG = {

    # --------------------------------------------------------
    # IMAGE
    # --------------------------------------------------------

    'img_size'        : 224,

    # --------------------------------------------------------
    # TRAINING
    # --------------------------------------------------------

    'batch_size'      : 32,
    'num_workers'     : 4,   # Kaggle stable
    'pin_memory'      : True,

    'epochs'          : 12,

    'lr'              : 1e-4,
    'weight_decay'    : 1e-4,

    # --------------------------------------------------------
    # FREEZE / UNFREEZE
    # --------------------------------------------------------

    'unfreeze_epoch'  : 3,
    'unfreeze_lr'     : 5e-6,

    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    'patience'        : 5,

    # --------------------------------------------------------
    # REGULARIZATION
    # --------------------------------------------------------

    'label_smoothing' : 0.10,

    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    'model_name'      : 'convnext_tiny',

    # --------------------------------------------------------
    # 13-CLASS SETUP
    # --------------------------------------------------------

    'num_classes'     : 13,

    'classes'         : [
        'SNE',
        'LY',
        'MO',
        'BL',
        'EO',
        'BA',
        'BNE',
        'VLY',
        'MY',
        'MMY',
        'PMY',
        'PC',
        'PLY'
    ],

    # --------------------------------------------------------
    # FOLDS
    # --------------------------------------------------------

    'n_folds'         : 3,

    # --------------------------------------------------------
    # KAGGLE OUTPUT DIRS
    # IMPORTANT:
    # /kaggle/working is writable
    # --------------------------------------------------------

    'ckpt_dir'        : '/kaggle/working/checkpoints',

    'save_dir'        : '/kaggle/working/checkpoints',

    # --------------------------------------------------------
    # CACHE DIR
    # --------------------------------------------------------

    'cache_dir'       : '/kaggle/working/wbc_cache'
}

# ============================================================
# CREATE DIRS
# ============================================================

os.makedirs(CFG['ckpt_dir'], exist_ok=True)
os.makedirs(CFG['save_dir'], exist_ok=True)
os.makedirs(CFG['cache_dir'], exist_ok=True)

print("\nCFG READY")

# ============================================================
# CLASS INDEX MAPPINGS
# ============================================================

IDX2CLASS = {

    0  : 'SNE',
    1  : 'LY',
    2  : 'MO',
    3  : 'BL',
    4  : 'EO',
    5  : 'BA',
    6  : 'BNE',
    7  : 'VLY',
    8  : 'MY',
    9  : 'MMY',
    10 : 'PMY',
    11 : 'PC',
    12 : 'PLY'
}

CLASS2IDX = {v: k for k, v in IDX2CLASS.items()}

print("\nIDX2CLASS:")
print(IDX2CLASS)

print("\nCLASS2IDX:")
print(CLASS2IDX)

# ============================================================
# IMPORTANT NOTES
# ============================================================

print("\nKAGGLE SETUP COMPLETE")
print("Using 13-class WBC classification")
print("ConvNeXt-Tiny configuration loaded")
print("Ready for denoise + ROI preprocessing")

In [ ]:
import pickle, numpy as np

oof_preds   = np.load('/kaggle/working/oof_preds.npy')
oof_targets = np.load('/kaggle/working/oof_targets.npy')

with open('/kaggle/working/fold_results.pkl', 'rb') as f:
    fold_results = pickle.load(f)

print("OOF results restored:")
print("  oof_preds   :", oof_preds.shape)
print("  oof_targets :", oof_targets.shape)
print("  fold_results:", len(fold_results), "folds")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 3 — LOAD WBCBench2026 CSVs (KAGGLE — FULLY FIXED)
# ══════════════════════════════════════════════════════════════════

import os
import cv2
import pandas as pd

BASE_PATH = "/kaggle/input/datasets/lasyapriya16/wbc-bench-2026"

print("BASE_PATH:", BASE_PATH)
print("\nFILES FOUND:")
for f in sorted(os.listdir(BASE_PATH)):
    print(" ", f)

# ── Load CSVs ─────────────────────────────────────────────────────
train_df = pd.read_csv(f"{BASE_PATH}/phase2_train.csv")
val_df   = pd.read_csv(f"{BASE_PATH}/phase2_eval.csv")
test_df  = pd.read_csv(f"{BASE_PATH}/phase2_test.csv")

print(f"\ntrain_df shape : {train_df.shape}")
print(f"val_df   shape : {val_df.shape}")
print(f"test_df  shape : {test_df.shape}")
print("\ntrain_df columns:", train_df.columns.tolist())

# ── Build label column (train + val only; test has no labels) ─────
train_df['label'] = train_df['labels'].astype(str).str.strip()
val_df['label']   = val_df['labels'].astype(str).str.strip()

# ── Build label_idx via CLASS2IDX ─────────────────────────────────
train_df['label_idx'] = train_df['label'].map(CLASS2IDX)
val_df['label_idx']   = val_df['label'].map(CLASS2IDX)

# Check for unmapped labels
for name, df in [('train_df', train_df), ('val_df', val_df)]:
    bad = df[df['label_idx'].isna()]
    if len(bad) > 0:
        print(f"\n[{name}] UNMAPPED LABELS FOUND:")
        print(bad['label'].value_counts())
        raise ValueError(
            f"[{name}] Labels not in CLASS2IDX: {bad['label'].unique()}\n"
            f"Expected: {list(CLASS2IDX.keys())}"
        )

train_df['label_idx'] = train_df['label_idx'].astype(int)
val_df['label_idx']   = val_df['label_idx'].astype(int)

# ── Build image paths ─────────────────────────────────────────────
train_df['path'] = train_df['ID'].apply(
    lambda x: os.path.join(BASE_PATH, 'phase2', 'train', str(x))
)
val_df['path'] = val_df['ID'].apply(
    lambda x: os.path.join(BASE_PATH, 'phase2', 'eval', str(x))
)
test_df['path'] = test_df['ID'].apply(
    lambda x: os.path.join(BASE_PATH, 'phase2', 'test', str(x))
)

# ── Drop rows with missing path / label / label_idx ───────────────
# IMPORTANT: reassign directly — not inside a loop with local 'df'
before = len(train_df)
train_df = train_df.dropna(subset=['path', 'label', 'label_idx']).reset_index(drop=True)
print(f"\ntrain_df: dropped {before - len(train_df)} NaN rows")

before = len(val_df)
val_df = val_df.dropna(subset=['path', 'label', 'label_idx']).reset_index(drop=True)
print(f"val_df  : dropped {before - len(val_df)} NaN rows")

before = len(test_df)
test_df = test_df.dropna(subset=['path']).reset_index(drop=True)
print(f"test_df : dropped {before - len(test_df)} NaN rows")

# ── Verify 13 classes ─────────────────────────────────────────────
print("\nUnique labels    :", sorted(train_df['label'].unique()))
print("Unique label_idx :", sorted(train_df['label_idx'].unique()))

n_found = train_df['label_idx'].nunique()
assert n_found == 13, (
    f"Expected 13 classes, found {n_found}.\n"
    f"Missing: {set(CLASS2IDX.keys()) - set(train_df['label'].unique())}"
)

# ── Final summary ──────────────────────────────────────────────────
print(f"\ntrain_df : {len(train_df):,} rows")
print(f"val_df   : {len(val_df):,} rows")
print(f"test_df  : {len(test_df):,} rows")

print("\nTRAIN CLASS DISTRIBUTION:")
dist = train_df['label'].value_counts()
for cls, count in dist.items():
    bar  = "█" * int(count / dist.max() * 25)
    flag = " ← minority" if count < 600 else ""
    print(f"  {cls:<6} {count:>7,}  {bar}{flag}")

print("\nCell 3 complete.")

In [ ]:
# ── Quick sanity check between Cell 3 and Cell 4 ─────────────────
print("train_df rows :", len(train_df))
print("val_df   rows :", len(val_df))
print("test_df  rows :", len(test_df))
print("train_df cols :", train_df.columns.tolist())
print("Sample path   :", train_df['path'].iloc[0])
print("Path exists   :", os.path.exists(train_df['path'].iloc[0]))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 4 — PATH VERIFICATION (FAST VERSION)
# Samples 50 random paths instead of checking all 30,000+
# ══════════════════════════════════════════════════════════════════

import os
import cv2

print(f"train_df : {len(train_df):,} rows")
print(f"val_df   : {len(val_df):,} rows")
print(f"test_df  : {len(test_df):,} rows")

if len(train_df) == 0:
    raise RuntimeError("train_df is empty — re-run Cell 3 first.")

# ── Spot-check 50 random paths (fast — not all 30k) ──────────────
sample_train = train_df['path'].sample(min(50, len(train_df)), random_state=42)
sample_val   = val_df['path'].sample(min(50, len(val_df)),   random_state=42)
sample_test  = test_df['path'].sample(min(50, len(test_df)), random_state=42)

train_ok = sample_train.apply(os.path.exists).sum()
val_ok   = sample_val.apply(os.path.exists).sum()
test_ok  = sample_test.apply(os.path.exists).sum()

print(f"\nSpot-check (50 samples each):")
print(f"  TRAIN : {train_ok}/50 valid")
print(f"  VAL   : {val_ok}/50 valid")
print(f"  TEST  : {test_ok}/50 valid")

if train_ok < 45:
    print("\nWARNING: Many paths missing — check BASE_PATH in Cell 3")
    print("Sample missing:", sample_train[~sample_train.apply(os.path.exists)].iloc[0])
else:
    print("\nAll paths look good.")

# ── Verify OpenCV can read one image ─────────────────────────────
sample_path = train_df['path'].iloc[0]
img = cv2.imread(sample_path)
print(f"\nSample path  : {sample_path}")
print(f"Image loaded : {img is not None}")
if img is not None:
    print(f"Image shape  : {img.shape}")
    print(f"Image dtype  : {img.dtype}")

print("\nCell 4 complete. Ready for augmentation and training.")

## Cell 3 — Mount Drive & load pre-saved splits

Your splits are already saved at `MyDrive/wbc/checkpoints/` from the previous session. We load them directly — no re-processing needed.

## Cell 4 — Fix image paths

The CSVs contain Drive paths. We re-point them to `/content/wbc` (local SSD) which is 10x faster for training.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 3b — Offline Minority Class Augmentation (KAGGLE VERSION)
#
# PLACEMENT: After Cell 3 + Cell 4 — BEFORE denoise/ROI cells
#
# Pipeline position:
#   Cell 3  : Load CSVs, build 13-class labels, build paths
#   Cell 4  : Path verification
#   Cell 3b : [THIS CELL] Augment minority classes → save to /kaggle/working
#   Cell 5  : Define denoise_wbc()
#   Cell 5b : Define crop_wbc_roi()
#   Cell 6  : Dataset class — runs denoise+crop on ALL images including these
# ══════════════════════════════════════════════════════════════════

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# ─────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────
# Kaggle /kaggle/working is writable — use it for augmented images
AUG_SAVE_DIR = '/kaggle/working/wbc_minority_aug'
os.makedirs(AUG_SAVE_DIR, exist_ok=True)

# Target counts based on your actual class distribution
AUG_TARGETS = {
    'PLY': 600,    # 11   → 600
    'PC' : 600,    # 56   → 600
    'PMY': 600,    # 76   → 600
    'VLY': 600,    # 282  → 600
    'BNE': 600,    # 340  → 600
    'MMY': 600,    # 335  → 600
    'MY' : 600,    # 405  → 600
    'BA' : 600,    # 420  → 600
    # EO (798), BL (1699), MO (2454), LY (6886), SNE (11135) — skip
}

print("Minority augmentation targets (based on your class counts):")
print(f"{'Class':<6} {'Current':>8} {'Target':>8} {'To Generate':>12}")
print("-" * 40)
for cls, tgt in AUG_TARGETS.items():
    current = int((train_df['label'] == cls).sum())
    needed  = max(0, tgt - current)
    print(f"  {cls:<6} {current:>8,} {tgt:>8,} {needed:>12,}")

print(f"\nAugmented images will be saved to: {AUG_SAVE_DIR}")


# ─────────────────────────────────────────────────────────────────
# LOAD + VALIDATE SOURCE IMAGE (raw, no denoise/crop)
# ─────────────────────────────────────────────────────────────────
def load_raw_source(path):
    """
    Load raw BGR image, convert to RGB.
    Returns uint8 RGB array or None if unreadable/blank.
    No denoise or crop — those run later in the Dataset.
    """
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        return None
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    if np.mean(gray) > 245 or np.mean(gray) < 5:
        return None
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


# ─────────────────────────────────────────────────────────────────
# AUGMENTATION FUNCTION
# Conservative — no hue shift (preserves EO/BA staining colour)
# No elastic distortion (preserves lobe/granule structure)
# ─────────────────────────────────────────────────────────────────
def augment_minority_image(img_rgb):
    """
    Conservative augmentation on uint8 RGB image.
    1. Random horizontal flip     (p=0.5)
    2. Random vertical flip       (p=0.5)
    3. Random 90° rotation        (0/90/180/270)
    4. Random ±25° rotation       (reflect border — no black corners)
    5. Mild brightness jitter     (×0.85–1.15)
    """
    img = img_rgb.astype(np.float32) / 255.0

    if np.random.rand() < 0.5:
        img = np.fliplr(img)

    if np.random.rand() < 0.5:
        img = np.flipud(img)

    img = np.rot90(img, np.random.randint(0, 4)).copy()

    angle  = np.random.uniform(-25, 25)
    h, w   = img.shape[:2]
    M      = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    img_u8 = (np.clip(img, 0, 1) * 255).astype(np.uint8)
    img_u8 = cv2.warpAffine(
        img_u8, M, (w, h),
        borderMode=cv2.BORDER_REFLECT_101
    )
    img = img_u8.astype(np.float32) / 255.0

    img = np.clip(img * np.random.uniform(0.85, 1.15), 0.0, 1.0)

    return (img * 255).astype(np.uint8)


# ─────────────────────────────────────────────────────────────────
# PREVIEW FUNCTION — verify quality before bulk generation
# ─────────────────────────────────────────────────────────────────
def preview_augmentation(df, cls_name, n_aug=4):
    cls_df   = df[df['label'] == cls_name].reset_index(drop=True)
    orig_rgb = None

    for _, row in cls_df.iterrows():
        candidate = load_raw_source(row['path'])
        if candidate is not None:
            orig_rgb = candidate
            break

    if orig_rgb is None:
        print(f"  [{cls_name}] No valid source image found for preview.")
        return

    fig, axes = plt.subplots(1, n_aug + 1, figsize=(3.5 * (n_aug + 1), 4))
    fig.suptitle(f'{cls_name} — Original + {n_aug} Augmented Variants',
                 fontsize=11, fontweight='bold')

    axes[0].imshow(orig_rgb)
    axes[0].set_title('Original', fontsize=9)
    axes[0].axis('off')

    for i in range(1, n_aug + 1):
        aug = augment_minority_image(orig_rgb.copy())
        axes[i].imshow(aug)
        axes[i].set_title(f'Aug {i}', fontsize=9)
        axes[i].axis('off')

    plt.tight_layout()
    preview_path = os.path.join(AUG_SAVE_DIR, f'aug_preview_{cls_name}.png')
    plt.savefig(preview_path, dpi=100, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f"  [{cls_name}] Preview saved → {preview_path}")


# ─────────────────────────────────────────────────────────────────
# BULK AUGMENTATION FUNCTION
# Saves raw PNGs to AUG_SAVE_DIR.
# Appends new rows to train_df and returns updated train_df.
# ─────────────────────────────────────────────────────────────────
def augment_class(train_df, cls_name, target_count, save_dir=AUG_SAVE_DIR):
    cls_df = train_df[train_df['label'] == cls_name].copy()
    n_have = len(cls_df)
    n_need = max(0, target_count - n_have)

    if n_need == 0:
        print(f"  [{cls_name}] Already {n_have} ≥ {target_count} — skipping.")
        return train_df

    print(f"  [{cls_name}] {n_have} → {target_count}  "
          f"(generating {n_need} images...)", end='', flush=True)

    # Load all valid source images for this class
    good_sources = []
    for _, row in cls_df.iterrows():
        img_rgb = load_raw_source(row['path'])
        if img_rgb is not None:
            good_sources.append((row, img_rgb))

    if not good_sources:
        print(f"\n  [{cls_name}] ERROR: No valid source images. Skipping.")
        return train_df

    new_rows    = []
    gen_idx     = 0
    attempts    = 0
    max_attempts = n_need * 8   # higher limit for PLY (only 11 sources)

    while gen_idx < n_need and attempts < max_attempts:
        for src_row, img_rgb in good_sources:
            if gen_idx >= n_need:
                break
            attempts += 1

            aug_rgb = augment_minority_image(img_rgb.copy())
            aug_bgr = cv2.cvtColor(aug_rgb, cv2.COLOR_RGB2BGR)

            # Reject if black border artefacts exceed 5% of pixels
            gray = cv2.cvtColor(aug_bgr, cv2.COLOR_BGR2GRAY)
            if np.sum(gray < 15) / gray.size > 0.05:
                continue

            fname = f"{cls_name}_aug_{gen_idx:06d}.png"
            fpath = os.path.join(save_dir, fname)
            cv2.imwrite(fpath, aug_bgr)

            # Copy all columns from source row, overwrite path/label/label_idx
            extra_cols = {
                c: src_row[c]
                for c in src_row.index
                if c not in ('path', 'label', 'label_idx')
            }
            new_rows.append({
                'path'      : fpath,
                'label'     : cls_name,
                'label_idx' : int(src_row['label_idx']),
                **extra_cols
            })
            gen_idx += 1

    if gen_idx < n_need:
        print(f"\n  [{cls_name}] WARNING: Only {gen_idx}/{n_need} generated.")
    else:
        print(f" done.")

    if not new_rows:
        return train_df

    aug_df       = pd.DataFrame(new_rows)
    train_df_aug = pd.concat([train_df, aug_df], ignore_index=True)
    train_df_aug = train_df_aug.sample(
        frac=1, random_state=SEED
    ).reset_index(drop=True)

    return train_df_aug


# ══════════════════════════════════════════════════════════════════
# STEP 1 — Preview augmentation quality for each minority class
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 1: Augmentation quality preview (1 sample per class)")
print("=" * 60)

for cls_name in AUG_TARGETS.keys():
    current = int((train_df['label'] == cls_name).sum())
    if current > 0:
        preview_augmentation(train_df, cls_name, n_aug=4)
    else:
        print(f"  [{cls_name}] Not found in train_df — check label names.")


# ══════════════════════════════════════════════════════════════════
# STEP 2 — Bulk augmentation for all minority classes
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 2: Bulk offline augmentation — all minority classes")
print("=" * 60)

for cls_name, target in AUG_TARGETS.items():
    train_df = augment_class(train_df, cls_name, target_count=target)


# ══════════════════════════════════════════════════════════════════
# STEP 3 — Final distribution check
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("STEP 3: Class distribution after augmentation")
print("=" * 60)
dist      = train_df['label'].value_counts()
max_count = dist.max()

print(f"\n{'Class':<8} {'Count':>8}  Bar")
print("-" * 55)
for cls, count in dist.items():
    bar  = "█" * int(count / max_count * 30)
    flag = " ← augmented" if cls in AUG_TARGETS else ""
    print(f"  {cls:<6} {count:>8,}  {bar}{flag}")

print(f"\nTotal train samples : {len(train_df):,}")
print(f"Augmented PNGs saved: {AUG_SAVE_DIR}")
print(f"Working dir contents: {len(os.listdir(AUG_SAVE_DIR))} files")
print("\nCell 3b complete.")
print("Next: Run denoise_wbc() cell, then crop_wbc_roi() cell.")
print("The Dataset class will automatically apply both to ALL images.")

Apply ROI cropping to extract nucleus region.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 5 — denoise_wbc() + crop_wbc_roi() + Visualization
# KAGGLE VERSION
#
# Output grid per class (4 columns):
#   Col 1: Original image
#   Col 2: Denoised image
#   Col 3: Original + RED tight bounding box overlay
#   Col 4: Context-aware ROI crop (what model sees)
# ══════════════════════════════════════════════════════════════════

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

try:
    from skimage.restoration import denoise_nl_means, estimate_sigma
    from skimage import img_as_float32, img_as_ubyte
    SKIMAGE_OK = True
    print("scikit-image found — NLM denoising enabled")
except ImportError:
    SKIMAGE_OK = False
    print("scikit-image missing — falling back to bilateral filter")


# ══════════════════════════════════════════════════════════════════
# 1. denoise_wbc()
# Stage 1: Non-Local Means  — kills Poisson/shot noise
# Stage 2: 3× tight bilateral — smooths cytoplasm, keeps edges sharp
# Stage 3: CLAHE on L-channel — boosts real contrast after denoising
# ══════════════════════════════════════════════════════════════════
def denoise_wbc(img_bgr):
    if img_bgr is None or img_bgr.size == 0:
        return img_bgr

    # Stage 1 — Non-Local Means
    if SKIMAGE_OK:
        img_rgb_f = img_as_float32(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        sigma_est = np.mean(estimate_sigma(img_rgb_f, channel_axis=-1))
        h_nlm     = float(np.clip(sigma_est * 1.0, 0.005, 0.08))
        nlm_rgb   = denoise_nl_means(
            img_rgb_f,
            h              = h_nlm,
            fast_mode      = True,
            patch_size     = 7,
            patch_distance = 11,
            channel_axis   = -1
        )
        denoised = cv2.cvtColor(
            img_as_ubyte(np.clip(nlm_rgb, 0, 1)),
            cv2.COLOR_RGB2BGR
        )
    else:
        denoised = cv2.bilateralFilter(img_bgr, d=11,
                                       sigmaColor=45, sigmaSpace=30)

    # Stage 2 — 3× anisotropic bilateral
    for _ in range(3):
        denoised = cv2.bilateralFilter(denoised, d=7,
                                       sigmaColor=25, sigmaSpace=15)

    # Stage 3 — CLAHE on L-channel
    lab      = cv2.cvtColor(denoised, cv2.COLOR_BGR2LAB)
    l, a, b  = cv2.split(lab)
    clahe    = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8))
    l_eq     = clahe.apply(l)
    denoised = cv2.cvtColor(cv2.merge([l_eq, a, b]), cv2.COLOR_LAB2BGR)
    return denoised


# ══════════════════════════════════════════════════════════════════
# 2. get_wbc_bbox()
# Returns (x1, y1, x2, y2) tight bounding box + padded square crop
# Separated from crop_wbc_roi so we can draw the box for visualization
# ══════════════════════════════════════════════════════════════════
def get_wbc_bbox(img_bgr, pad_frac=0.15):
    """
    Detects WBC using 4-method cascade.
    Returns (x1, y1, x2, y2) of the padded square bounding box,
    or None if all methods fail (fallback will be used).
    """
    if img_bgr is None or img_bgr.size == 0:
        return None

    h, w     = img_bgr.shape[:2]
    img_area = h * w

    # Extra smooth for robust detection on noisy images
    smooth = cv2.bilateralFilter(img_bgr, d=11, sigmaColor=55, sigmaSpace=55)
    smooth = cv2.GaussianBlur(smooth, (5, 5), sigmaX=1.2)

    hsv          = cv2.cvtColor(smooth, cv2.COLOR_BGR2HSV)
    lab          = cv2.cvtColor(smooth, cv2.COLOR_BGR2LAB)
    _, s_ch, v_ch = cv2.split(hsv)
    l_ch, a_ch, b_ch = cv2.split(lab)
    gray         = cv2.cvtColor(smooth, cv2.COLOR_BGR2GRAY)

    def mask_to_bbox(mask, min_frac=0.02, max_frac=0.75):
        k_c = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (13, 13))
        k_o = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_c, iterations=2)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k_o, iterations=1)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            return None
        valid = [c for c in contours
                 if img_area * min_frac <= cv2.contourArea(c) <= img_area * max_frac]
        if not valid:
            valid = [max(contours, key=cv2.contourArea)]
        best        = max(valid, key=cv2.contourArea)
        x, y, bw, bh = cv2.boundingRect(best)

        # Add contextual padding
        px = int(bw * pad_frac);  py = int(bh * pad_frac)
        x1 = max(0, x - px);     y1 = max(0, y - py)
        x2 = min(w, x + bw + px); y2 = min(h, y + bh + py)

        # Make square
        side = max(x2 - x1, y2 - y1)
        cx   = (x1 + x2) // 2;  cy = (y1 + y2) // 2
        x1   = max(0, cx - side // 2)
        y1   = max(0, cy - side // 2)
        x2   = min(w, x1 + side)
        y2   = min(h, y1 + side)
        if x2 - x1 < side: x1 = max(0, x2 - side)
        if y2 - y1 < side: y1 = max(0, y2 - side)

        # Reject if not meaningfully smaller than full image
        if (x2 - x1) * (y2 - y1) > img_area * 0.88:
            return None
        return (x1, y1, x2, y2)

    # Method 1 — Otsu on inverted Value (dark WBC vs bright background)
    v_inv        = 255 - v_ch
    _, mask_otsu = cv2.threshold(v_inv, 0, 255,
                                  cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    median_v     = np.median(v_ch)
    mask_dark    = (v_ch < median_v * 0.92).astype(np.uint8) * 255
    bbox = mask_to_bbox(cv2.bitwise_and(mask_otsu, mask_dark))
    if bbox: return bbox

    # Method 2 — HSV purple/violet nucleus
    mask_hsv = cv2.inRange(hsv,
                            np.array([105, 30, 20],  dtype=np.uint8),
                            np.array([170, 255, 215], dtype=np.uint8))
    bbox = mask_to_bbox(mask_hsv)
    if bbox: return bbox

    # Method 3 — LAB b* + a* cytoplasm cue
    _, mb = cv2.threshold(b_ch, 125, 255, cv2.THRESH_BINARY_INV)
    _, ma = cv2.threshold(a_ch, 120, 255, cv2.THRESH_BINARY)
    bbox  = mask_to_bbox(cv2.bitwise_and(mb, ma))
    if bbox: return bbox

    # Method 4 — Adaptive threshold (noisy images)
    mask_ada = cv2.adaptiveThreshold(gray, 255,
                                      cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                      cv2.THRESH_BINARY_INV,
                                      blockSize=35, C=8)
    bbox = mask_to_bbox(mask_ada, min_frac=0.03, max_frac=0.70)
    if bbox: return bbox

    return None   # all methods failed → caller uses fallback


# ══════════════════════════════════════════════════════════════════
# 3. crop_wbc_roi()
# Uses get_wbc_bbox() then crops. Falls back to centre 60% if needed.
# ══════════════════════════════════════════════════════════════════
def crop_wbc_roi(img_bgr, pad_frac=0.15):
    """
    Context-aware whole-cell morphology ROI crop.
    Preserves: nucleus + cytoplasm + granules + membrane + RBC context.
    """
    if img_bgr is None or img_bgr.size == 0:
        return img_bgr

    h, w = img_bgr.shape[:2]
    bbox = get_wbc_bbox(img_bgr, pad_frac=pad_frac)

    if bbox is not None:
        x1, y1, x2, y2 = bbox
        crop = img_bgr[y1:y2, x1:x2]
        if crop is not None and crop.size > 0 \
                and crop.shape[0] >= 32 and crop.shape[1] >= 32:
            return crop

    # Final fallback — centre 60% square
    side = int(min(h, w) * 0.60)
    x1   = max(0, w // 2 - side // 2)
    y1   = max(0, h // 2 - side // 2)
    return img_bgr[y1:y1 + side, x1:x1 + side]


# ══════════════════════════════════════════════════════════════════
# 4. visualize_all_classes()
# 13 rows × 4 columns
# Col 1: Original
# Col 2: Denoised
# Col 3: Original + RED bounding box (tight WBC detection)
# Col 4: Context-aware ROI crop (what the model sees)
# ══════════════════════════════════════════════════════════════════
CLASS_FULL = {
    'SNE': 'Segmented Neutrophil', 'LY' : 'Lymphocyte',
    'MO' : 'Monocyte',            'BL' : 'Blast',
    'EO' : 'Eosinophil',          'BA' : 'Basophil',
    'BNE': 'Band Neutrophil',     'VLY': 'Variant Lymphocyte',
    'MY' : 'Myelocyte',           'MMY': 'Metamyelocyte',
    'PMY': 'Promyelocyte',        'PC' : 'Plasma Cell',
    'PLY': 'Prolymphocyte',
}

# Biological feature preserved per class — shown in Col 4
MORPH_BADGE = {
    'SNE': 'Lobes + granules',    'LY' : 'High N:C ratio',
    'MO' : 'Large cytoplasm',     'BL' : 'Blast chromatin',
    'EO' : 'Orange granules',     'BA' : 'Dark granules',
    'BNE': 'Band nucleus',        'VLY': 'Variant chromatin',
    'MY' : 'Myelocyte gran.',     'MMY': 'Indented nucleus',
    'PMY': 'Prominent nucleoli',  'PC' : 'Clock-face nucleus',
    'PLY': 'Prolymph chromatin',
}

COL_TITLES = [
    'Original',
    'Denoised\n(NLM → Bilateral → CLAHE)',
    'Tight Bounding Box\n(RED box = WBC detection)',
    'Context-Aware ROI Crop\n(model input — 15% padding)',
]

def visualize_all_classes(df, save_dir=None, dpi=130):
    """
    For each of 13 WBC classes, shows:
      Col 1: Original image
      Col 2: After denoise_wbc()
      Col 3: Original with RED tight bounding box drawn on it
      Col 4: Final context-aware ROI crop (what ConvNeXt sees)
    """
    ALL_CLASSES = CFG['classes']
    n_rows      = len(ALL_CLASSES)

    fig = plt.figure(figsize=(4.2 * 4, 3.5 * n_rows))
    gs  = gridspec.GridSpec(
        n_rows, 4, figure=fig,
        hspace=0.20, wspace=0.05,
        left=0.13, right=0.995,
        top=0.970, bottom=0.008
    )

    print(f"Processing {n_rows} classes...")

    for row_i, cls_name in enumerate(ALL_CLASSES):

        # ── Pick one clean readable sample ───────────────────────
        img_bgr     = None
        cls_rows    = df[df['label'] == cls_name].sample(
            frac=1, random_state=42
        ).reset_index(drop=True)

        for _, r in cls_rows.head(40).iterrows():
            cand = cv2.imread(r['path'])
            if cand is None or cand.size == 0:
                continue
            gcheck = cv2.cvtColor(cand, cv2.COLOR_BGR2GRAY)
            if float(np.mean(gcheck)) > 248 or float(np.mean(gcheck)) < 5:
                continue
            img_bgr = cand
            break

        if img_bgr is not None:
            den_bgr  = denoise_wbc(img_bgr)
            crop_bgr = crop_wbc_roi(den_bgr)
            bbox     = get_wbc_bbox(den_bgr)

            orig_rgb = cv2.cvtColor(img_bgr,  cv2.COLOR_BGR2RGB)
            den_rgb  = cv2.cvtColor(den_bgr,  cv2.COLOR_BGR2RGB)
            crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)

            # ── Draw RED bounding box on a copy of the original ──
            bbox_img = orig_rgb.copy()
            if bbox is not None:
                x1, y1, x2, y2 = bbox
                # Thick red rectangle
                cv2.rectangle(bbox_img, (x1, y1), (x2, y2),
                              color=(220, 30, 30), thickness=3)
                # Corner ticks (like sir's reference image)
                tick = max(8, (x2 - x1) // 8)
                for (px, py) in [(x1,y1),(x2,y1),(x1,y2),(x2,y2)]:
                    dx = tick  if px == x1 else -tick
                    dy = tick  if py == y1 else -tick
                    cv2.line(bbox_img, (px, py), (px+dx, py),
                             (255, 220, 0), 2)
                    cv2.line(bbox_img, (px, py), (px, py+dy),
                             (255, 220, 0), 2)
                # Label inside box
                cv2.putText(
                    bbox_img,
                    f'+{int(0.15*100)}% pad',
                    (x1 + 4, y2 - 6),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.45, (255, 255, 255), 1, cv2.LINE_AA
                )
            else:
                # No detection — show fallback label
                cv2.putText(bbox_img, 'fallback crop',
                            (10, 20), cv2.FONT_HERSHEY_SIMPLEX,
                            0.5, (255, 100, 100), 1, cv2.LINE_AA)

            # Stats
            ho, wo   = orig_rgb.shape[:2]
            hc, wc   = crop_rgb.shape[:2]
            pct      = 100.0 * (hc * wc) / (ho * wo)
            noise_m  = float(np.mean(cv2.absdiff(
                cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY),
                cv2.cvtColor(den_bgr, cv2.COLOR_BGR2GRAY)
            )))

            print(f"  [{cls_name:<5}] "
                  f"orig={wo}×{ho}  "
                  f"crop={wc}×{hc}  "
                  f"coverage={pct:.0f}%  "
                  f"noise_removed={noise_m:.1f}  "
                  f"bbox={'detected' if bbox else 'fallback'}")

            images = [orig_rgb, den_rgb, bbox_img, crop_rgb]

        else:
            blank  = np.full((224, 224, 3), 40, dtype=np.uint8)
            images = [blank, blank, blank, blank]
            pct    = 0.0
            wc, hc = 224, 224
            print(f"  [{cls_name:<5}] WARNING: no readable image found")

        # ── Draw 4 panels ─────────────────────────────────────────
        for col_i, im in enumerate(images):
            ax = fig.add_subplot(gs[row_i, col_i])
            ax.imshow(im, interpolation='bilinear')
            ax.set_xticks([])
            ax.set_yticks([])

            # Column headers on row 0 only
            if row_i == 0:
                ax.set_title(COL_TITLES[col_i],
                             fontsize=8, fontweight='bold', pad=5)

            # Row label on col 0
            if col_i == 0:
                ax.set_ylabel(
                    f'{cls_name}\n{CLASS_FULL[cls_name]}',
                    fontsize=7.5, rotation=0,
                    labelpad=72, ha='right', va='center'
                )

            # Crop size annotation on col 3
            if col_i == 3 and img_bgr is not None:
                ax.text(
                    0.03, 0.03,
                    f'{wc}×{hc}px  {pct:.0f}%\n'
                    f'{MORPH_BADGE.get(cls_name,"")}',
                    transform=ax.transAxes,
                    fontsize=5.5, color='cyan', va='bottom',
                    bbox=dict(boxstyle='round,pad=0.2',
                              facecolor='black', alpha=0.65)
                )

            # Bounding box method annotation on col 2
            if col_i == 2 and img_bgr is not None:
                method_txt = 'Detected' if bbox else 'Fallback 60%'
                ax.text(
                    0.03, 0.97, method_txt,
                    transform=ax.transAxes,
                    fontsize=6, color='lime', va='top',
                    bbox=dict(boxstyle='round,pad=0.2',
                              facecolor='black', alpha=0.65)
                )

    fig.suptitle(
        'WBC-Bench 2026 — All 13 Classes — Preprocessing Pipeline\n'
        'denoise_wbc(): NLM + Bilateral + CLAHE   |   '
        'crop_wbc_roi(): Context-Aware Tight Bounding Box (RED) + 15% Padding',
        fontsize=9, fontweight='bold', y=0.999
    )

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        out = os.path.join(save_dir, 'all13_roi_visualization.png')
        plt.savefig(out, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f"\n  PNG saved → {out}")

    plt.show()
    plt.close(fig)
    print("Visualization complete.")


# ══════════════════════════════════════════════════════════════════
# SANITY CHECK — 5 samples, print original vs crop size
# ══════════════════════════════════════════════════════════════════
print("=" * 62)
print("Sanity check: denoise_wbc() + crop_wbc_roi() on 5 samples")
print("=" * 62)
_ok = 0
for i in range(min(50, len(train_df))):
    _row  = train_df.iloc[i]
    _img  = cv2.imread(_row['path'])
    if _img is None:
        continue
    _den  = denoise_wbc(_img)
    _crop = crop_wbc_roi(_den)
    _H, _W   = _img.shape[:2]
    _ch, _cw = _crop.shape[:2]
    _pct     = 100.0 * (_ch * _cw) / (_H * _W)
    _red     = 100.0 - _pct
    _bbox    = get_wbc_bbox(_den)
    print(f"  [{_row['label']:<5}] "
          f"orig={_W}×{_H}  "
          f"crop={_cw}×{_ch}  "
          f"coverage={_pct:.0f}%  "
          f"reduction={_red:.0f}%  "
          f"bbox={'OK' if _bbox else 'fallback'}")
    _ok += 1
    if _ok >= 5:
        break

if _ok == 0:
    print("  ERROR: No images readable — check paths in train_df")
else:
    print(f"\n  denoise_wbc()  : OK")
    print(f"  crop_wbc_roi() : OK")
    print(f"  get_wbc_bbox() : OK")

# ══════════════════════════════════════════════════════════════════
# RUN FULL VISUALIZATION — all 13 classes
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 62)
print("Running full 13-class visualization...")
print("(~30–60 seconds — one sample per class)")
print("=" * 62)

visualize_all_classes(train_df, save_dir=CFG['save_dir'])

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 5 — denoise_wbc() + crop_wbc_roi() + Visualization
# KAGGLE VERSION — FIXED
#
# Problems fixed vs previous version:
#   1. NLM denoising was too weak for extreme noise → replaced with
#      stronger multi-stage pipeline specifically for microscopy noise
#   2. ROI detection ran on still-noisy images → now uses heavily
#      pre-smoothed internal copy ONLY for detection (not for crop)
#   3. Bounding box colour was orange due to BGR/RGB confusion → fixed
#   4. Fallback was 60% centre → raised to 70%, still clearly smaller
#   5. No RBC context in Col 4 → padding raised to 20% for noisy imgs
# ══════════════════════════════════════════════════════════════════

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline


# ══════════════════════════════════════════════════════════════════
# 1. denoise_wbc()
# Specifically tuned for Wright-Giemsa stained WBC microscopy.
# Key insight: run STRONG denoise first, THEN CLAHE.
# Running CLAHE before denoise amplifies noise — that was the bug.
#
# Stage 1: FastNlMeansDenoisingColored (OpenCV built-in, no skimage)
#          h=10, hColor=10 — aggressive shot/Poisson noise removal
# Stage 2: 2× bilateral filter — edge-preserving smoothing
#          keeps nucleus/granule/lobe boundaries sharp
# Stage 3: CLAHE on L-channel — contrast boost AFTER noise is gone
# ══════════════════════════════════════════════════════════════════
def denoise_wbc(img_bgr):
    if img_bgr is None or img_bgr.size == 0:
        return img_bgr

    # Stage 1 — OpenCV NLM (always available, no skimage needed)
    # h=10 is aggressive — needed for extreme microscopy noise
    denoised = cv2.fastNlMeansDenoisingColored(
        img_bgr,
        None,
        h         = 10,    # luminance noise filter strength
        hColor    = 10,    # colour noise filter strength
        templateWindowSize = 7,
        searchWindowSize   = 21
    )

    # Stage 2 — 2× edge-preserving bilateral
    for _ in range(2):
        denoised = cv2.bilateralFilter(
            denoised, d=9, sigmaColor=35, sigmaSpace=25
        )

    # Stage 3 — CLAHE on L-channel (AFTER noise removal)
    lab      = cv2.cvtColor(denoised, cv2.COLOR_BGR2LAB)
    l, a, b  = cv2.split(lab)
    clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_eq     = clahe.apply(l)
    denoised = cv2.cvtColor(cv2.merge([l_eq, a, b]),
                             cv2.COLOR_LAB2BGR)
    return denoised


# ══════════════════════════════════════════════════════════════════
# 2. get_wbc_bbox()
# Key fix: uses an EXTRA-heavily smoothed copy for detection only.
# The actual crop is taken from img_bgr (the denoised image).
# This separates "detection quality" from "crop quality".
#
# Detection strategy:
#   The WBC is ALWAYS darker than the bright pink/white RBC background.
#   → Invert the Value channel → WBC becomes the brightest blob
#   → Otsu threshold → finds it reliably even with noise
#   This works across ALL 13 classes because it doesn't rely on
#   specific stain colour — just relative brightness.
# ══════════════════════════════════════════════════════════════════
def get_wbc_bbox(img_bgr, pad_frac=0.20):
    """
    Returns (x1, y1, x2, y2) tight padded square bounding box.
    Returns None if detection fails (caller uses fallback).

    pad_frac=0.20 (20%) ensures cytoplasm + RBC context included.
    """
    if img_bgr is None or img_bgr.size == 0:
        return None

    h, w     = img_bgr.shape[:2]
    img_area = h * w

    # ── Detection copy: extra-heavy smooth for robust thresholding ─
    # This is ONLY used for finding contours — NOT for the actual crop
    det = cv2.GaussianBlur(img_bgr, (15, 15), sigmaX=3.0)
    det = cv2.bilateralFilter(det, d=15, sigmaColor=80, sigmaSpace=80)

    hsv          = cv2.cvtColor(det, cv2.COLOR_BGR2HSV)
    lab          = cv2.cvtColor(det, cv2.COLOR_BGR2LAB)
    _, s_ch, v_ch = cv2.split(hsv)
    l_ch, a_ch, b_ch = cv2.split(lab)
    gray         = cv2.cvtColor(det, cv2.COLOR_BGR2GRAY)

    # ── Shared helper ──────────────────────────────────────────────
    def mask_to_bbox(mask, min_frac=0.015, max_frac=0.80):
        """Convert binary mask → padded square bounding box coords."""
        k_c = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17, 17))
        k_o = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7,  7))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_c, iterations=3)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k_o, iterations=1)

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            return None

        valid = [c for c in contours
                 if img_area * min_frac
                    <= cv2.contourArea(c)
                    <= img_area * max_frac]
        if not valid:
            valid = [max(contours, key=cv2.contourArea)]

        best         = max(valid, key=cv2.contourArea)
        x, y, bw, bh = cv2.boundingRect(best)

        # Contextual padding
        px = int(bw * pad_frac);   py = int(bh * pad_frac)
        x1 = max(0, x - px);      y1 = max(0, y - py)
        x2 = min(w, x + bw + px); y2 = min(h, y + bh + py)

        # Make square
        side = max(x2 - x1, y2 - y1)
        cx   = (x1 + x2) // 2;    cy = (y1 + y2) // 2
        x1   = max(0, cx - side // 2)
        y1   = max(0, cy - side // 2)
        x2   = min(w, x1 + side)
        y2   = min(h, y1 + side)
        if x2 - x1 < side: x1 = max(0, x2 - side)
        if y2 - y1 < side: y1 = max(0, y2 - side)

        # Must be meaningfully smaller than full image
        if (x2 - x1) * (y2 - y1) > img_area * 0.90:
            return None
        return (x1, y1, x2, y2)

    # ── Method 1: Otsu on inverted Value (primary — most robust) ──
    # WBC is always darker than background → invert makes it bright
    v_inv        = 255 - v_ch
    _, mask_otsu = cv2.threshold(v_inv, 0, 255,
                                  cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # Gate: only keep pixels darker than 90% of median brightness
    median_v  = np.median(v_ch)
    mask_dark = (v_ch < median_v * 0.90).astype(np.uint8) * 255
    m1        = cv2.bitwise_and(mask_otsu, mask_dark)
    bbox      = mask_to_bbox(m1)
    if bbox: return bbox

    # ── Method 2: HSV purple/violet (nucleus stain) ───────────────
    mask_hsv = cv2.inRange(
        hsv,
        np.array([100, 25, 15],  dtype=np.uint8),
        np.array([175, 255, 220], dtype=np.uint8)
    )
    bbox = mask_to_bbox(mask_hsv)
    if bbox: return bbox

    # ── Method 3: LAB colour (cytoplasm cue) ─────────────────────
    _, mb = cv2.threshold(b_ch, 128, 255, cv2.THRESH_BINARY_INV)
    _, ma = cv2.threshold(a_ch, 118, 255, cv2.THRESH_BINARY)
    bbox  = mask_to_bbox(cv2.bitwise_and(mb, ma))
    if bbox: return bbox

    # ── Method 4: Saturation threshold (last resort) ─────────────
    _, ms = cv2.threshold(s_ch, 30, 255, cv2.THRESH_BINARY)
    bbox  = mask_to_bbox(ms, min_frac=0.02, max_frac=0.75)
    if bbox: return bbox

    # ── Method 5: Adaptive threshold on gray ─────────────────────
    mask_ada = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        blockSize=41, C=6
    )
    bbox = mask_to_bbox(mask_ada, min_frac=0.02, max_frac=0.75)
    if bbox: return bbox

    return None


# ══════════════════════════════════════════════════════════════════
# 3. crop_wbc_roi()
# Applies get_wbc_bbox() and returns the crop.
# Fallback: centre 70% square (always visibly smaller than original).
# ══════════════════════════════════════════════════════════════════
def crop_wbc_roi(img_bgr, pad_frac=0.20):
    if img_bgr is None or img_bgr.size == 0:
        return img_bgr

    h, w = img_bgr.shape[:2]
    bbox = get_wbc_bbox(img_bgr, pad_frac=pad_frac)

    if bbox is not None:
        x1, y1, x2, y2 = bbox
        crop = img_bgr[y1:y2, x1:x2]
        if crop is not None and crop.size > 0 \
                and crop.shape[0] >= 32 and crop.shape[1] >= 32:
            return crop

    # Fallback — centre 70% square
    side = int(min(h, w) * 0.70)
    x1   = max(0, w // 2 - side // 2)
    y1   = max(0, h // 2 - side // 2)
    return img_bgr[y1:y1 + side, x1:x1 + side]


# ══════════════════════════════════════════════════════════════════
# 4. visualize_all_classes()
# 13 rows × 4 columns
#   Col 1: Original (raw image as-is)
#   Col 2: After denoise_wbc()
#   Col 3: Denoised + RED tight bounding box
#   Col 4: Context-aware ROI crop (model input)
#
# Box drawn on DENOISED image (Col 3) so it's visible without noise
# ══════════════════════════════════════════════════════════════════
CLASS_FULL = {
    'SNE':'Segmented Neutrophil', 'LY':'Lymphocyte',
    'MO':'Monocyte',              'BL':'Blast',
    'EO':'Eosinophil',            'BA':'Basophil',
    'BNE':'Band Neutrophil',      'VLY':'Variant Lymphocyte',
    'MY':'Myelocyte',             'MMY':'Metamyelocyte',
    'PMY':'Promyelocyte',         'PC':'Plasma Cell',
    'PLY':'Prolymphocyte',
}

MORPH_BADGE = {
    'SNE':'Lobes + granules',   'LY':'High N:C ratio',
    'MO':'Large cytoplasm',     'BL':'Blast chromatin',
    'EO':'Orange granules',     'BA':'Dark granules',
    'BNE':'Band nucleus',       'VLY':'Variant chromatin',
    'MY':'Myelocyte gran.',     'MMY':'Indented nucleus',
    'PMY':'Prominent nucleoli', 'PC':'Clock-face nucleus',
    'PLY':'Prolymph chromatin',
}

COL_TITLES = [
    'Original\n(raw)',
    'Denoised\n(NLM → Bilateral → CLAHE)',
    'Tight Bounding Box\n(RED = detected WBC region)',
    'Context-Aware ROI Crop\n(model input — 20% padding)',
]

def draw_red_bbox(img_rgb, bbox):
    """Draw red bounding box + yellow corner ticks on RGB image."""
    out = img_rgb.copy()
    if bbox is None:
        cv2.putText(out, 'fallback', (8, 22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                    (255, 80, 80), 1, cv2.LINE_AA)
        return out

    x1, y1, x2, y2 = bbox

    # Red rectangle — drawn in RGB so (255,30,30) is correct red
    cv2.rectangle(out, (x1, y1), (x2, y2),
                  color=(220, 30, 30), thickness=3)

    # Yellow corner ticks
    tick = max(10, (x2 - x1) // 7)
    for (px, py) in [(x1,y1),(x2,y1),(x1,y2),(x2,y2)]:
        dx = tick  if px == x1 else -tick
        dy = tick  if py == y1 else -tick
        cv2.line(out, (px, py), (px + dx, py), (255, 220, 0), 2)
        cv2.line(out, (px, py), (px, py + dy), (255, 220, 0), 2)

    # White label inside bottom of box
    label = '+20% pad'
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX,
                                   0.42, 1)
    cv2.rectangle(out, (x1+2, y2-th-6), (x1+tw+6, y2-1),
                  (0, 0, 0), -1)
    cv2.putText(out, label, (x1+4, y2-4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.42,
                (255, 255, 255), 1, cv2.LINE_AA)
    return out


def visualize_all_classes(df, save_dir=None, dpi=130):
    ALL_CLASSES = CFG['classes']
    n_rows      = len(ALL_CLASSES)

    fig = plt.figure(figsize=(4.5 * 4, 3.8 * n_rows))
    gs  = gridspec.GridSpec(
        n_rows, 4, figure=fig,
        hspace=0.18, wspace=0.04,
        left=0.13, right=0.998,
        top=0.972, bottom=0.005
    )

    print(f"Processing {n_rows} classes — one sample each...")

    for row_i, cls_name in enumerate(ALL_CLASSES):

        # Pick one clean sample for this class
        img_bgr  = None
        cls_rows = df[df['label'] == cls_name].sample(
            frac=1, random_state=42
        ).reset_index(drop=True)

        for _, r in cls_rows.head(60).iterrows():
            cand = cv2.imread(r['path'])
            if cand is None or cand.size == 0:
                continue
            gcheck = cv2.cvtColor(cand, cv2.COLOR_BGR2GRAY)
            mean_v = float(np.mean(gcheck))
            if mean_v > 249 or mean_v < 3:
                continue
            img_bgr = cand
            break

        if img_bgr is not None:
            den_bgr  = denoise_wbc(img_bgr)
            bbox     = get_wbc_bbox(den_bgr)
            crop_bgr = crop_wbc_roi(den_bgr)

            # Convert BGR → RGB for matplotlib
            orig_rgb = cv2.cvtColor(img_bgr,  cv2.COLOR_BGR2RGB)
            den_rgb  = cv2.cvtColor(den_bgr,  cv2.COLOR_BGR2RGB)
            crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)

            # Red bounding box drawn on DENOISED image
            # (so box is visible — not hidden by noise)
            bbox_rgb = draw_red_bbox(den_rgb, bbox)

            ho, wo = orig_rgb.shape[:2]
            hc, wc = crop_rgb.shape[:2]
            pct    = 100.0 * (hc * wc) / (ho * wo)

            print(f"  [{cls_name:<5}] "
                  f"orig={wo}×{ho}  "
                  f"crop={wc}×{hc}  "
                  f"area={pct:.0f}%  "
                  f"{'DETECTED' if bbox else 'FALLBACK'}")

            images = [orig_rgb, den_rgb, bbox_rgb, crop_rgb]

        else:
            blank  = np.full((224, 224, 3), 50, dtype=np.uint8)
            images = [blank, blank, blank, blank]
            wc, hc, pct = 224, 224, 100.0
            print(f"  [{cls_name:<5}] WARNING: no readable image found")

        for col_i, im in enumerate(images):
            ax = fig.add_subplot(gs[row_i, col_i])
            ax.imshow(im, interpolation='bilinear')
            ax.set_xticks([])
            ax.set_yticks([])

            # Column headers — row 0 only
            if row_i == 0:
                ax.set_title(COL_TITLES[col_i],
                             fontsize=8.5, fontweight='bold', pad=5)

            # Row label — col 0 only
            if col_i == 0:
                ax.set_ylabel(
                    f'{cls_name}\n{CLASS_FULL[cls_name]}',
                    fontsize=8, rotation=0,
                    labelpad=74, ha='right', va='center'
                )

            # Crop annotation — col 3
            if col_i == 3 and img_bgr is not None:
                ax.text(
                    0.03, 0.03,
                    f'{wc}×{hc}px  {pct:.0f}%\n'
                    f'{MORPH_BADGE.get(cls_name,"")}',
                    transform=ax.transAxes,
                    fontsize=5.5, color='cyan', va='bottom',
                    bbox=dict(boxstyle='round,pad=0.2',
                              facecolor='black', alpha=0.70)
                )

            # Detection status — col 2
            if col_i == 2 and img_bgr is not None:
                status = 'Detected ✓' if bbox else 'Fallback 70%'
                color  = 'lime'       if bbox else 'orange'
                ax.text(
                    0.03, 0.97, status,
                    transform=ax.transAxes,
                    fontsize=6.5, color=color, va='top',
                    bbox=dict(boxstyle='round,pad=0.2',
                              facecolor='black', alpha=0.70)
                )

    fig.suptitle(
        'WBC-Bench 2026 — All 13 Classes — Full Preprocessing Pipeline\n'
        'Col 1: Raw  |  Col 2: Denoised (NLM+Bilateral+CLAHE)  |  '
        'Col 3: Tight RED Bounding Box  |  Col 4: Context-Aware ROI (20% padding)',
        fontsize=9, fontweight='bold', y=0.999
    )

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        out = os.path.join(save_dir, 'all13_roi_visualization.png')
        plt.savefig(out, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f"\n  Saved → {out}")

    plt.show()
    plt.close(fig)
    print("Done.")


# ══════════════════════════════════════════════════════════════════
# SANITY CHECK — 5 samples
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("Sanity check: denoise_wbc() + get_wbc_bbox() + crop_wbc_roi()")
print("=" * 65)
_ok = 0
for i in range(min(80, len(train_df))):
    _row = train_df.iloc[i]
    _img = cv2.imread(_row['path'])
    if _img is None:
        continue
    _den  = denoise_wbc(_img)
    _crop = crop_wbc_roi(_den)
    _bbox = get_wbc_bbox(_den)
    _H, _W   = _img.shape[:2]
    _ch, _cw = _crop.shape[:2]
    _pct     = 100.0 * (_ch * _cw) / (_H * _W)
    print(f"  [{_row['label']:<5}] "
          f"orig={_W}×{_H}  "
          f"crop={_cw}×{_ch}  "
          f"area={_pct:.0f}%  "
          f"{'DETECTED' if _bbox else 'FALLBACK'}")
    _ok += 1
    if _ok >= 5:
        break

print(f"\n  denoise_wbc()  : OK")
print(f"  crop_wbc_roi() : OK")
print(f"  get_wbc_bbox() : OK")

# ══════════════════════════════════════════════════════════════════
# RUN FULL VISUALIZATION
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("Running 13-class visualization (~60–90 sec)...")
print("=" * 65)

visualize_all_classes(train_df, save_dir=CFG['save_dir'])

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 5b — Class-Wise ROI Visualization (Professor Presentation)
#
# PLACEMENT: After Cell 5 (denoise + ROI functions defined)
#
# Shows 13 rows × 3 columns:
#   Col 1 — Original image (raw)
#   Col 2 — Denoised + RED tight bounding box overlay
#   Col 3 — Context-aware ROI crop (what the model sees)
#
# Uses get_wbc_bbox() and crop_wbc_roi() from Cell 5 directly.
# No duplicate detection logic here.
# ══════════════════════════════════════════════════════════════════

import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

FULL_NAMES = {
    'SNE': 'Segmented Neutrophil', 'LY' : 'Lymphocyte',
    'MO' : 'Monocyte',             'BL' : 'Blast',
    'EO' : 'Eosinophil',           'BA' : 'Basophil',
    'BNE': 'Band Neutrophil',      'VLY': 'Variant Lymphocyte',
    'MY' : 'Myelocyte',            'MMY': 'Metamyelocyte',
    'PMY': 'Promyelocyte',         'PC' : 'Plasma Cell',
    'PLY': 'Prolymphocyte',
}

MORPH_BADGE = {
    'SNE': 'Lobes + granules',    'LY' : 'Sparse cytoplasm',
    'MO' : 'Large cytoplasm',     'BL' : 'Blast chromatin',
    'EO' : 'Orange granules',     'BA' : 'Dark granules',
    'BNE': 'Band nucleus',        'VLY': 'Variant chromatin',
    'MY' : 'Myelocyte gran.',     'MMY': 'Indented nucleus',
    'PMY': 'Prominent nucleoli',  'PC' : 'Clock-face nucleus',
    'PLY': 'Prolymph chromatin',
}

def draw_bbox_on_image(img_rgb, bbox):
    """
    Draws RED bounding box + yellow corner ticks on RGB image.
    Returns annotated copy.
    """
    out = img_rgb.copy()

    if bbox is None:
        cv2.putText(out, 'fallback crop', (8, 24),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                    (255, 80, 80), 1, cv2.LINE_AA)
        return out

    x1, y1, x2, y2 = bbox

    # RED rectangle (RGB colour space — correct)
    cv2.rectangle(out, (x1, y1), (x2, y2),
                  color=(220, 30, 30), thickness=3)

    # Yellow corner ticks
    tick = max(10, (x2 - x1) // 7)
    for (px, py) in [(x1,y1), (x2,y1), (x1,y2), (x2,y2)]:
        dx = tick  if px == x1 else -tick
        dy = tick  if py == y1 else -tick
        cv2.line(out, (px, py), (px + dx, py), (255, 215, 0), 2)
        cv2.line(out, (px, py), (px, py + dy), (255, 215, 0), 2)

    # Padding label inside bottom of box
    label = '+20% contextual padding'
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.40, 1)
    cv2.rectangle(out, (x1+2, y2-th-7), (x1+tw+6, y2-1),
                  (0, 0, 0), -1)
    cv2.putText(out, label, (x1+4, y2-4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.40,
                (255, 255, 255), 1, cv2.LINE_AA)
    return out


def visualize_roi_all_classes(df, save_path=None):
    """
    13 rows × 3 columns:
      Col 1: Original raw image
      Col 2: Denoised image + RED tight bounding box
      Col 3: Context-aware ROI crop (model input)

    Verifies for each class:
      ✓ Cytoplasm visible in Col 3
      ✓ RBC context at crop edges
      ✓ Tight box in Col 2 covers whole cell
      ✓ Granules/lobes/membrane preserved
    """
    classes   = CFG['classes']
    n_classes = len(classes)

    fig, axes = plt.subplots(
        n_classes, 3,
        figsize=(3 * 4.5, n_classes * 4.0)
    )

    fig.suptitle(
        'Experiment-1: Context-Aware Whole-Cell ROI — All 13 WBC Classes\n'
        'Col 1: Original  |  Col 2: RED Tight Bounding Box  |  '
        'Col 3: Context-Aware ROI Crop (model input)',
        fontsize=12, fontweight='bold', y=1.001
    )

    # Column headers
    col_titles = [
        'Original Image',
        'Tight Bounding Box\n(RED = detected WBC)',
        'Context-Aware ROI Crop\n(nucleus + cytoplasm + RBC context)'
    ]
    for j, title in enumerate(col_titles):
        axes[0, j].set_title(title, fontsize=10,
                              fontweight='bold', pad=8)

    for row_i, cls_name in enumerate(classes):

        # ── Find one valid sample for this class ──────────────────
        cls_df  = df[df['label'] == cls_name].sample(
            frac=1, random_state=42
        ).reset_index(drop=True)

        img_bgr = None
        den_bgr = None
        bbox    = None
        crop_bgr = None

        for _, sample_row in cls_df.head(60).iterrows():
            candidate = cv2.imread(sample_row['path'])
            if candidate is None or candidate.size == 0:
                continue
            gray_check = cv2.cvtColor(candidate, cv2.COLOR_BGR2GRAY)
            mean_v = float(np.mean(gray_check))
            if mean_v > 249 or mean_v < 3:
                continue

            den  = denoise_wbc(candidate)
            bb   = get_wbc_bbox(den)       # uses Cell 5 function
            crop = crop_wbc_roi(den)       # uses Cell 5 function

            img_bgr  = candidate
            den_bgr  = den
            bbox     = bb
            crop_bgr = crop
            break

        # ── Blank row if no image found ───────────────────────────
        if img_bgr is None:
            for j in range(3):
                axes[row_i, j].set_facecolor('#111122')
                axes[row_i, j].text(
                    0.5, 0.5,
                    f'No valid {cls_name}\nimage found',
                    ha='center', va='center',
                    color='white', fontsize=9,
                    transform=axes[row_i, j].transAxes
                )
                axes[row_i, j].axis('off')
            axes[row_i, 0].set_ylabel(
                f'{cls_name}\n{FULL_NAMES.get(cls_name,"")}',
                fontsize=9, fontweight='bold',
                rotation=0, labelpad=60, va='center'
            )
            continue

        # ── Convert to RGB for matplotlib ────────────────────────
        orig_rgb = cv2.cvtColor(img_bgr,  cv2.COLOR_BGR2RGB)
        den_rgb  = cv2.cvtColor(den_bgr,  cv2.COLOR_BGR2RGB)
        crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)

        # Draw RED bbox on DENOISED image (visible without noise)
        bbox_rgb = draw_bbox_on_image(den_rgb, bbox)

        ho, wo = orig_rgb.shape[:2]
        hc, wc = crop_rgb.shape[:2]
        pct    = 100.0 * (hc * wc) / (ho * wo)

        images = [orig_rgb, bbox_rgb, crop_rgb]

        # ── Plot 3 panels ─────────────────────────────────────────
        for col_i, im in enumerate(images):
            ax = axes[row_i, col_i]
            ax.imshow(im, interpolation='bilinear')
            ax.set_xticks([])
            ax.set_yticks([])

            # Row label on col 0
            if col_i == 0:
                ax.set_ylabel(
                    f'{cls_name}\n{FULL_NAMES.get(cls_name,"")}',
                    fontsize=8.5, fontweight='bold',
                    rotation=0, labelpad=62, va='center'
                )

            # Detection status on col 1
            if col_i == 1:
                status = 'Detected ✓' if bbox else 'Fallback 70%'
                color  = 'lime'       if bbox else 'orange'
                ax.text(
                    0.03, 0.97, status,
                    transform=ax.transAxes,
                    fontsize=7, color=color, va='top',
                    bbox=dict(boxstyle='round,pad=0.2',
                              facecolor='black', alpha=0.70)
                )

            # Crop size + morphology badge on col 2
            if col_i == 2:
                ax.text(
                    0.03, 0.03,
                    f'{wc}×{hc}px  {pct:.0f}% area',
                    transform=ax.transAxes,
                    fontsize=6.5, color='cyan', va='bottom',
                    bbox=dict(boxstyle='round,pad=0.2',
                              facecolor='black', alpha=0.70)
                )
                ax.text(
                    0.50, 0.98,
                    MORPH_BADGE.get(cls_name, ''),
                    transform=ax.transAxes,
                    fontsize=6.5, color='yellow',
                    ha='center', va='top',
                    bbox=dict(boxstyle='round,pad=0.2',
                              facecolor='black', alpha=0.60)
                )

        # Print progress
        print(f"  [{cls_name:<5}] {FULL_NAMES.get(cls_name,''):<25} "
              f"crop={wc}×{hc}  area={pct:.0f}%  "
              f"{'DETECTED' if bbox else 'FALLBACK'}")

    plt.tight_layout(rect=[0, 0, 1, 0.999])

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=130, bbox_inches='tight',
                    facecolor='white')
        print(f"\n  Saved → {save_path}")

    plt.show()
    plt.close(fig)


# ── Run ───────────────────────────────────────────────────────────
print("=" * 65)
print("Class-Wise ROI Visualization — All 13 WBC Classes")
print("One sample per class | ~30–60 seconds")
print("=" * 65)

viz_save = os.path.join(CFG['save_dir'], 'experiment1_roi_all_classes.png')
visualize_roi_all_classes(train_df, save_path=viz_save)

print("\nVerify for each class:")
print("  ✓ Col 3 shows cytoplasm (not nucleus-only crop)")
print("  ✓ Col 3 shows some surrounding RBC context at edges")
print("  ✓ Col 2 RED box covers the WHOLE cell, not just nucleus")
print("  ✓ EO/BA/BNE/SNE show granule-containing cytoplasm in Col 3")

In [ ]:
 visualize_all_classes(train_df, save_dir=CFG['save_dir'])

## Cell 5 — Transforms

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 6 — Transforms (KAGGLE VERSION)
# ══════════════════════════════════════════════════════════════════

from torchvision import transforms

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
SZ   = CFG['img_size']   # 224

# ── Standard train augmentation (majority classes) ────────────────
# SNE, LY, MO, BL, EO — enough samples, moderate augmentation
train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((SZ, SZ)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    # NO hue shift — preserves Wright-Giemsa staining colours
    # EO orange granules, BA dark granules depend on exact hue
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ── Strong augmentation for minority classes ──────────────────────
# BA, BNE, VLY, MY, MMY, PMY, PC, PLY — very few samples
# Stronger geometric + brightness augmentation
# NO hue shift — would corrupt EO/BA granule colour cues
minority_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((SZ, SZ)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(45),
    transforms.ColorJitter(
        brightness = 0.4,
        contrast   = 0.4,
        saturation = 0.3,
        # hue deliberately omitted — staining colour is diagnostic
    ),
    transforms.RandomAffine(
        degrees    = 0,
        translate  = (0.12, 0.12),
        scale      = (0.82, 1.18)
    ),
    transforms.RandomPerspective(distortion_scale=0.25, p=0.5),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.2)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ── Val / test — no augmentation ─────────────────────────────────
val_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((SZ, SZ)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ── Minority class set ────────────────────────────────────────────
# Based on your actual Kaggle class counts (all < 800 samples)
MINORITY_CLASSES = {
    'BA',   # 420
    'BNE',  # 340
    'VLY',  # 282
    'MY',   # 405
    'MMY',  # 335
    'PMY',  # 76
    'PC',   # 56
    'PLY',  # 11
    'EO',   # 798  — borderline, include for safety
}

# ── Verify ────────────────────────────────────────────────────────
print("train_tf    defined:", train_tf    is not None)
print("minority_tf defined:", minority_tf is not None)
print("val_tf      defined:", val_tf      is not None)
print(f"\nMinority classes ({len(MINORITY_CLASSES)}):")
for cls in sorted(MINORITY_CLASSES):
    count = int((train_df['label'] == cls).sum())
    print(f"  {cls:<6} : {count:>6,} samples → minority_tf")
print("\nMajority classes → train_tf:")
for cls in CFG['classes']:
    if cls not in MINORITY_CLASSES:
        count = int((train_df['label'] == cls).sum())
        print(f"  {cls:<6} : {count:>6,} samples → train_tf")

print("\nTransforms ready.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 7 — WBCDataset class (KAGGLE VERSION)
# Pipeline per image:
#   Load → Denoise → ROI Crop → BGR→RGB → Augment → Normalise
# ══════════════════════════════════════════════════════════════════

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

class WBCDataset(Dataset):
    """
    Loads WBC images on-the-fly.
    Pipeline per image:
        Load → denoise_wbc() → crop_wbc_roi() → RGB → transform → tensor
    """
    def __init__(self, df, mode='train'):
        self.df   = df.reset_index(drop=True)
        self.mode = mode   # 'train', 'val', 'test'

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ── Step 1: Load ──────────────────────────────────────────
        img_bgr = cv2.imread(row['path'])
        if img_bgr is None:
            img_bgr = np.full((SZ, SZ, 3), 128, dtype=np.uint8)

        # ── Step 2: Denoise ───────────────────────────────────────
        try:
            img_bgr = denoise_wbc(img_bgr)
        except Exception:
            pass   # keep original if denoise fails

        # ── Step 3: ROI Crop ──────────────────────────────────────
        try:
            img_bgr = crop_wbc_roi(img_bgr)
        except Exception:
            pass   # keep denoised image if crop fails

        # ── Step 4: BGR → RGB ─────────────────────────────────────
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # ── Step 5: Augment + Normalise ───────────────────────────
        if self.mode == 'train':
            if row['label'] in MINORITY_CLASSES:
                tensor = minority_tf(img_rgb)
            else:
                tensor = train_tf(img_rgb)
        else:
            # val and test both use val_tf (no augmentation)
            tensor = val_tf(img_rgb)

        # ── Step 6: Label ─────────────────────────────────────────
        # test_df has no label_idx — return -1 as placeholder
        if 'label_idx' in row.index and not pd.isna(row['label_idx']):
            label = torch.tensor(int(row['label_idx']), dtype=torch.long)
        else:
            label = torch.tensor(-1, dtype=torch.long)

        return tensor, label


# ── Sanity check ──────────────────────────────────────────────────
print("Checking dependencies...")
print(f"  train_tf      : {'OK' if 'train_tf'      in dir() else 'MISSING — run Cell 6'}")
print(f"  minority_tf   : {'OK' if 'minority_tf'   in dir() else 'MISSING — run Cell 6'}")
print(f"  val_tf        : {'OK' if 'val_tf'         in dir() else 'MISSING — run Cell 6'}")
print(f"  denoise_wbc   : {'OK' if 'denoise_wbc'   in dir() else 'MISSING — run Cell 5'}")
print(f"  crop_wbc_roi  : {'OK' if 'crop_wbc_roi'  in dir() else 'MISSING — run Cell 5'}")
print(f"  MINORITY_CLASSES: {'OK' if 'MINORITY_CLASSES' in dir() else 'MISSING — run Cell 6'}")
print(f"  SZ            : {SZ}")
print(f"  IDX2CLASS     : {'OK' if 'IDX2CLASS' in dir() else 'MISSING — run Cell 2'}")

# ── Test on 4 train samples ───────────────────────────────────────
print("\nTesting WBCDataset on 4 train samples...")
ds = WBCDataset(train_df.head(4), mode='train')

for i in range(len(ds)):
    img, lbl = ds[i]
    cls_name = IDX2CLASS.get(lbl.item(), f'idx={lbl.item()}')
    print(f"  [{i}] shape={tuple(img.shape)}  "
          f"label={lbl.item()} ({cls_name})  "
          f"range=[{img.min():.2f}, {img.max():.2f}]  "
          f"tf={'minority' if train_df.iloc[i]['label'] in MINORITY_CLASSES else 'train'}")

# ── Test on 2 val samples ─────────────────────────────────────────
print("\nTesting WBCDataset on 2 val samples...")
ds_val = WBCDataset(val_df.head(2), mode='val')
for i in range(len(ds_val)):
    img, lbl = ds_val[i]
    cls_name = IDX2CLASS.get(lbl.item(), f'idx={lbl.item()}')
    print(f"  [{i}] shape={tuple(img.shape)}  "
          f"label={lbl.item()} ({cls_name})  "
          f"range=[{img.min():.2f}, {img.max():.2f}]")

# ── Test on 2 test samples (no labels) ───────────────────────────
print("\nTesting WBCDataset on 2 test samples (no labels expected)...")
ds_test = WBCDataset(test_df.head(2), mode='test')
for i in range(len(ds_test)):
    img, lbl = ds_test[i]
    print(f"  [{i}] shape={tuple(img.shape)}  "
          f"label={lbl.item()} (-1 = no label, expected for test)  "
          f"range=[{img.min():.2f}, {img.max():.2f}]")

print("\nWBCDataset ready.")
print("Pipeline confirmed: Load → Denoise → ROI Crop → RGB → Augment → Tensor")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# REPLACEMENT CELL 19 — HybridLoss + make_criterion()
# KAGGLE VERSION
#
# PLACE  : Replace Cell 19 entirely (the FocalLoss + make_criterion cell)
# NOTHING ELSE CHANGES:
#   - Cell 32 CV loop       still calls: criterion = make_criterion(f_tr)
#   - Cell 34 final model   still calls: criterion = make_criterion(trainval_df)
#   - All other cells       untouched
#
# ADD ONE LINE to CFG in Cell 2 (if not already present):
#   'hybrid_alpha' : 0.5,
#
# WHY HYBRID OVER PURE FOCAL:
#   Pure Focal with gamma=2 over-focuses on noisy hard samples
#   (augmented BA, PLY, PC minority images) → unstable gradients
#   → overconfident predictions → OOF=0.77 but Test=0.21 gap.
#
#   Hybrid = 0.5 × CrossEntropy  +  0.5 × FocalLoss
#   CE     → stable gradients, prevents oscillation
#   Focal  → class balancing, down-weights easy majority classes
#   Both carry label smoothing + per-class alpha weights
# ══════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import numpy as np
from sklearn.utils.class_weight import compute_class_weight


class HybridLoss(nn.Module):
    """
    Hybrid Loss = mix * CrossEntropy + (1 - mix) * FocalLoss

    Both CE and Focal share:
      - soft targets from label smoothing (prevents overconfidence)
      - per-class alpha weights (handles class imbalance)

    Parameters
    ----------
    alpha     : Tensor  shape (C,) per-class weights from compute_class_weight
    gamma     : float   focal focusing parameter (default 2.0)
    mix       : float   weight on CE part; (1-mix) = Focal weight (default 0.5)
    smoothing : float   label smoothing epsilon (default 0.1)
    """
    def __init__(self, alpha=None, gamma=2.0, mix=0.5, smoothing=0.1):
        super().__init__()
        self.alpha     = alpha
        self.gamma     = gamma
        self.mix       = mix
        self.smoothing = smoothing

    def forward(self, logits, targets):
        """
        logits  : (N, C)  raw model output — NOT softmaxed
        targets : (N,)    integer class indices
        """
        n_cls = logits.size(1)
        log_p = torch.nn.functional.log_softmax(logits, dim=1)  # (N, C)
        p     = log_p.exp()                                      # (N, C)

        # ── Soft targets (label smoothing) ────────────────────────
        # true class  → 1 - smoothing
        # other classes → smoothing / (C - 1)
        with torch.no_grad():
            soft = torch.full_like(logits, self.smoothing / (n_cls - 1))
            soft.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)

        # ── CE part: standard cross-entropy against soft targets ──
        ce_loss = -(soft * log_p).sum(dim=1)                     # (N,)

        # ── Focal part: focal weight from hard target probability ─
        pt           = p.gather(1, targets.unsqueeze(1)).squeeze(1)  # (N,)
        focal_weight = (1.0 - pt) ** self.gamma                      # (N,)
        focal_loss   = -(soft * log_p).sum(dim=1) * focal_weight     # (N,)

        # ── Weighted combination ──────────────────────────────────
        loss = self.mix * ce_loss + (1.0 - self.mix) * focal_loss    # (N,)

        # ── Per-class alpha weighting ─────────────────────────────
        if self.alpha is not None:
            alpha_t = self.alpha.to(logits.device)[targets]           # (N,)
            loss    = alpha_t * loss

        return loss.mean()


def make_criterion(df, gamma=2.0):
    """
    Builds HybridLoss with per-class inverse-frequency weights.

    Identical signature to the old make_criterion — no changes needed
    anywhere else in the notebook.

    Parameters
    ----------
    df    : pd.DataFrame  training fold df (must have 'label_idx' column)
    gamma : float         focal gamma (default 2.0)

    Returns
    -------
    HybridLoss  ready to use as criterion in train_one_epoch / evaluate
    """
    y     = df['label_idx'].values.astype(int)
    w     = compute_class_weight('balanced', classes=np.unique(y), y=y)
    alpha = torch.tensor(w, dtype=torch.float32)

    mix       = CFG.get('hybrid_alpha',    0.5)
    smoothing = CFG.get('label_smoothing', 0.1)

    criterion = HybridLoss(
        alpha     = alpha,
        gamma     = gamma,
        mix       = mix,
        smoothing = smoothing,
    )

    print(f"  HybridLoss ready")
    print(f"    CE weight      : {mix:.2f}  (stable gradients)")
    print(f"    Focal weight   : {1.0 - mix:.2f}  (class balancing)")
    print(f"    Gamma          : {gamma}")
    print(f"    Label smooth   : {smoothing}")
    print(f"    Class weights  :")
    for i, (cls, wi) in enumerate(zip(CFG['classes'], w)):
        print(f"      {cls:<5} → {wi:.4f}")

    return criterion


# ── Quick self-test ───────────────────────────────────────────────
print("=" * 55)
print("HybridLoss self-test...")
_dummy_logits  = torch.randn(8, 13)
_dummy_targets = torch.randint(0, 13, (8,))
_dummy_alpha   = torch.ones(13)
_hl = HybridLoss(alpha=_dummy_alpha, gamma=2.0, mix=0.5, smoothing=0.1)
_loss = _hl(_dummy_logits, _dummy_targets)
assert _loss.item() > 0, "Loss should be positive"
assert not torch.isnan(_loss), "Loss should not be NaN"
print(f"  Test loss = {_loss.item():.4f}  ✓")
print("=" * 55)
print()
print("HybridLoss class   : ready")
print("make_criterion()   : ready")
print()
print("REQUIRED — add to CFG dict in Cell 2 if not present:")
print("  'hybrid_alpha' : 0.5,   # 0.5 = 50/50 CE + Focal")
print()
print("Usage (unchanged from before):")
print("  criterion = make_criterion(train_df)")
print("  criterion = make_criterion(f_tr)   # inside CV fold loop")

In [ ]:
# ════════════════════════════════════════════════════════════════
# FAST PREPROCESS + CACHE VERSION
# KAGGLE OPTIMIZED
# ════════════════════════════════════════════════════════════════

import os
import cv2
import hashlib
import numpy as np

from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

CACHE_DIR = "/kaggle/working/wbc_cache"

os.makedirs(CACHE_DIR, exist_ok=True)

IMG_SIZE = 224

# ============================================================
# FAST WORKER
# ============================================================

def _process_one(args):

    orig_path, cache_dir = args

    h = hashlib.md5(
        orig_path.encode()
    ).hexdigest()[:12]

    dest_path = os.path.join(
        cache_dir,
        f"c_{h}.jpg"
    )

    # --------------------------------------------------------
    # skip if already cached
    # --------------------------------------------------------

    if os.path.exists(dest_path):

        chk = cv2.imread(dest_path)

        if chk is not None and chk.size > 0:

            return (
                orig_path,
                dest_path,
                'skipped'
            )

    # --------------------------------------------------------
    # read image
    # --------------------------------------------------------

    img = cv2.imread(orig_path)

    if img is None:

        return (
            orig_path,
            orig_path,
            'failed'
        )

    try:

        # ====================================================
        # FAST DENOISE
        # ====================================================

        img = cv2.medianBlur(img, 3)

        img = cv2.bilateralFilter(
            img,
            d=7,
            sigmaColor=45,
            sigmaSpace=45
        )

        # ====================================================
        # CLAHE
        # ====================================================

        lab = cv2.cvtColor(
            img,
            cv2.COLOR_BGR2LAB
        )

        l, a, b = cv2.split(lab)

        clahe = cv2.createCLAHE(
            clipLimit=2.0,
            tileGridSize=(8,8)
        )

        l = clahe.apply(l)

        img = cv2.cvtColor(
            cv2.merge([l,a,b]),
            cv2.COLOR_LAB2BGR
        )

        # ====================================================
        # MILD SHARPEN
        # ====================================================

        blur = cv2.GaussianBlur(
            img,
            (0,0),
            1.0
        )

        img = cv2.addWeighted(
            img,
            1.12,
            blur,
            -0.12,
            0
        )

        # ====================================================
        # ROI
        # ====================================================

        H, W = img.shape[:2]

        hsv = cv2.cvtColor(
            img,
            cv2.COLOR_BGR2HSV
        )

        lab = cv2.cvtColor(
            img,
            cv2.COLOR_BGR2LAB
        )

        gray = cv2.cvtColor(
            img,
            cv2.COLOR_BGR2GRAY
        )

        _, _, v = cv2.split(hsv)

        _, a2, b2 = cv2.split(lab)

        # ----------------------------------------------------
        # mask
        # ----------------------------------------------------

        v_inv = 255 - v

        _, mask1 = cv2.threshold(
            v_inv,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

        _, mask2 = cv2.threshold(
            b2,
            125,
            255,
            cv2.THRESH_BINARY_INV
        )

        mask = cv2.bitwise_and(
            mask1,
            mask2
        )

        # ----------------------------------------------------
        # morphology
        # ----------------------------------------------------

        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (9,9)
        )

        mask = cv2.morphologyEx(
            mask,
            cv2.MORPH_CLOSE,
            kernel,
            iterations=2
        )

        # ----------------------------------------------------
        # contours
        # ----------------------------------------------------

        cnts, _ = cv2.findContours(
            mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        if len(cnts) > 0:

            c = max(
                cnts,
                key=cv2.contourArea
            )

            x, y, bw, bh = cv2.boundingRect(c)

            # tighter padding
            px = int(bw * 0.07)
            py = int(bh * 0.07)

            x1 = max(0, x-px)
            y1 = max(0, y-py)

            x2 = min(W, x+bw+px)
            y2 = min(H, y+bh+py)

            crop = img[y1:y2, x1:x2]

        else:

            # fallback centre crop
            side = int(min(H,W) * 0.60)

            x1 = W//2 - side//2
            y1 = H//2 - side//2

            crop = img[
                y1:y1+side,
                x1:x1+side
            ]

        # ====================================================
        # resize
        # ====================================================

        crop = cv2.resize(
            crop,
            (IMG_SIZE, IMG_SIZE),
            interpolation=cv2.INTER_AREA
        )

        # ====================================================
        # save
        # ====================================================

        cv2.imwrite(
            dest_path,
            crop,
            [cv2.IMWRITE_JPEG_QUALITY, 95]
        )

        return (
            orig_path,
            dest_path,
            'cached'
        )

    except Exception:

        return (
            orig_path,
            orig_path,
            'failed'
        )

# ============================================================
# CACHE FUNCTION
# ============================================================

def preprocess_and_cache(
    df,
    cache_dir=CACHE_DIR,
    n_workers=2
):

    df = df.copy()

    args = [
        (p, cache_dir)
        for p in df['path'].tolist()
    ]

    path_map = {}

    n_cached = 0
    n_skipped = 0
    n_failed = 0

    print(f"\nCaching {len(df):,} images")

    with ProcessPoolExecutor(
        max_workers=n_workers
    ) as pool:

        futures = {
            pool.submit(_process_one, a): a[0]
            for a in args
        }

        with tqdm(
            total=len(args),
            desc="Cache",
            unit="img"
        ) as pbar:

            for fut in as_completed(futures):

                orig, dest, status = fut.result()

                path_map[orig] = dest

                if status == 'cached':
                    n_cached += 1

                elif status == 'skipped':
                    n_skipped += 1

                else:
                    n_failed += 1

                pbar.update(1)

    df['path'] = df['path'].map(path_map)

    print(f"\nCached : {n_cached}")
    print(f"Skipped: {n_skipped}")
    print(f"Failed : {n_failed}")

    return df

# ============================================================
# IMPORTANT TEST FIRST
# DO NOT CACHE FULL DATASET IMMEDIATELY
# ============================================================

DEBUG = True

if DEBUG:

    print("\nDEBUG MODE")

    train_df_debug = train_df.head(100)

    train_df_debug = preprocess_and_cache(
        train_df_debug,
        n_workers=2
    )

    print("\n100 IMAGE DEBUG CACHE COMPLETE")

else:

    train_df = preprocess_and_cache(
        train_df,
        n_workers=2
    )

    val_df = preprocess_and_cache(
        val_df,
        n_workers=2
    )

    test_df = preprocess_and_cache(
        test_df,
        n_workers=2
    )

    print("\nFULL CACHE COMPLETE")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 7 — WBCDataset (FAST CACHED VERSION)
# Images already denoised + cropped + resized in cache step.
# __getitem__ just loads → augments → normalizes. No heavy processing.
# ══════════════════════════════════════════════════════════════════

import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.utils.class_weight import compute_class_weight


class WBCDataset(Dataset):
    def __init__(self, df, mode='train'):
        self.df   = df.reset_index(drop=True)
        self.mode = mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ── Load cached image (already denoised + cropped) ────────
        img_bgr = cv2.imread(row['path'])
        if img_bgr is None:
            img_bgr = np.full((SZ, SZ, 3), 128, dtype=np.uint8)

        # ── BGR → RGB ─────────────────────────────────────────────
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # ── Augment + Normalise ───────────────────────────────────
        if self.mode == 'train':
            if 'label' in row.index and row['label'] in MINORITY_CLASSES:
                tensor = minority_tf(img_rgb)
            else:
                tensor = train_tf(img_rgb)
        else:
            tensor = val_tf(img_rgb)

        # ── Label ─────────────────────────────────────────────────
        if 'label_idx' in row.index and not pd.isna(row['label_idx']):
            label = torch.tensor(int(row['label_idx']), dtype=torch.long)
        else:
            label = torch.tensor(-1, dtype=torch.long)

        return tensor, label


def make_weighted_sampler(df):
    counts         = df['label_idx'].value_counts().sort_index().values.astype(float)
    weights        = 1.0 / counts
    sample_weights = torch.tensor(
        [weights[int(i)] for i in df['label_idx']], dtype=torch.float
    )
    return WeightedRandomSampler(
        weights     = sample_weights,
        num_samples = len(sample_weights),
        replacement = True
    )


def make_loaders(tr_df, va_df, te_df):
    tr_ds = WBCDataset(tr_df, mode='train')
    va_ds = WBCDataset(va_df, mode='val')
    te_ds = WBCDataset(te_df, mode='test')

    sampler = make_weighted_sampler(tr_df)

    tr_loader = DataLoader(
        tr_ds,
        batch_size  = CFG['batch_size'],
        sampler     = sampler,
        num_workers = CFG['num_workers'],
        pin_memory  = CFG['pin_memory'],
        drop_last   = True
    )
    va_loader = DataLoader(
        va_ds,
        batch_size  = CFG['batch_size'],
        shuffle     = False,
        num_workers = CFG['num_workers'],
        pin_memory  = CFG['pin_memory']
    )
    te_loader = DataLoader(
        te_ds,
        batch_size  = CFG['batch_size'],
        shuffle     = False,
        num_workers = CFG['num_workers'],
        pin_memory  = CFG['pin_memory']
    )
    return tr_loader, va_loader, te_loader


def get_class_weights(df):
    y = df['label_idx'].values.astype(int)
    w = compute_class_weight('balanced', classes=np.unique(y), y=y)
    return torch.tensor(w, dtype=torch.float).to(device)


# ── Sanity check ──────────────────────────────────────────────────
print("Sanity check WBCDataset (cached)...")
_ds = WBCDataset(train_df.head(4), mode='train')
for i in range(4):
    _img, _lbl = _ds[i]
    print(f"  [{i}] shape={tuple(_img.shape)}  "
          f"label={_lbl.item()}  "
          f"range=[{_img.min():.2f},{_img.max():.2f}]")
print("Dataset / sampler / loader helpers ready.")

## Cell 6 — Dataset class

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CACHE CELL — Preprocesses ALL images in 2–5 minutes
# Uses fast bilateral-only denoising (no NLM — too slow for caching)
# ══════════════════════════════════════════════════════════════════

import os
import cv2
import numpy as np
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

CACHE_DIR = '/kaggle/working/img_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

# ── Fast denoiser for caching (completes in 2–5 min total) ───────
# fastNlMeans = 0.3 sec/image × 46k = 3.8 hrs  ← too slow
# fast_denoise = 0.003 sec/image × 46k = 2.3 min ← use this
def fast_denoise_and_crop(img_bgr):
    """
    Fast preprocessing for caching:
    - Bilateral filter ×2 (edge-preserving, ~3ms per image)
    - CLAHE on L-channel
    - crop_wbc_roi()
    - resize 224×224
    NLM is intentionally skipped here — too slow for bulk caching.
    """
    if img_bgr is None or img_bgr.size == 0:
        return np.full((224, 224, 3), 128, dtype=np.uint8)

    # Fast bilateral (not NLM)
    d = cv2.bilateralFilter(img_bgr, d=7, sigmaColor=35, sigmaSpace=25)
    d = cv2.bilateralFilter(d,       d=7, sigmaColor=35, sigmaSpace=25)

    # CLAHE on L
    lab     = cv2.cvtColor(d, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    d       = cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]),
                            cv2.COLOR_LAB2BGR)

    # ROI crop
    try:
        d = crop_wbc_roi(d)
    except Exception:
        pass

    return cv2.resize(d, (224, 224))


def process_one(args):
    """Process and cache one image. Returns (new_path, error_or_None)."""
    row, split_name = args
    fname    = f"{split_name}_{os.path.basename(row['path'])}"
    out_path = os.path.join(CACHE_DIR, fname)

    if os.path.exists(out_path):
        return out_path, None

    img = cv2.imread(row['path'])
    if img is None:
        img = np.full((224, 224, 3), 128, dtype=np.uint8)
    else:
        img = fast_denoise_and_crop(img)

    cv2.imwrite(out_path, img)
    return out_path, None


def preprocess_and_cache(df, split_name, n_workers=4):
    """
    Caches all images using thread pool for parallel I/O.
    n_workers=4 matches CFG['num_workers'].
    """
    args      = [(row, split_name) for _, row in df.iterrows()]
    new_paths = [None] * len(args)
    skipped   = 0

    with ThreadPoolExecutor(max_workers=n_workers) as exe:
        futures = {exe.submit(process_one, a): i
                   for i, a in enumerate(args)}
        for fut in tqdm(as_completed(futures),
                        total=len(futures),
                        desc=f'Caching {split_name}'):
            i              = futures[fut]
            path, err      = fut.result()
            new_paths[i]   = path
            if err:
                print(f"  Warning: {err}")

    already = sum(1 for a in args
                  if os.path.exists(
                      os.path.join(CACHE_DIR,
                          f"{split_name}_{os.path.basename(a[0]['path'])}")
                  ))
    print(f"  {split_name}: {len(new_paths)} total  "
          f"({already} were already cached)")
    return new_paths


# ── Run ───────────────────────────────────────────────────────────
print("=" * 60)
print("Caching all images (fast bilateral denoising)...")
print("Expected: 2–5 minutes total")
print("=" * 60)

import time
t0 = time.time()

train_cached = preprocess_and_cache(train_df, 'train', n_workers=4)
val_cached   = preprocess_and_cache(val_df,   'val',   n_workers=4)
test_cached  = preprocess_and_cache(test_df,  'test',  n_workers=4)

elapsed = time.time() - t0
print(f"\nTotal caching time: {elapsed/60:.1f} minutes")

# Update paths
train_df = train_df.copy(); train_df['path'] = train_cached
val_df   = val_df.copy();   val_df['path']   = val_cached
test_df  = test_df.copy();  test_df['path']  = test_cached

# Verify
n_ok = sum(os.path.exists(p) for p in train_df['path'].head(20))
print(f"Verification: {n_ok}/20 sample paths exist")
print(f"Cache dir: {len(os.listdir(CACHE_DIR)):,} files")
print("\nDone. Run Cell 7 then Cell 11.")

In [ ]:
# ── Cell 6 — Dataset class + weighted sampler + loaders ──────────

class WBCDataset(Dataset):
    def __init__(self, df, mode='train'):
        self.df   = df.reset_index(drop=True)
        self.mode = mode

    def __len__(self):
        return len(self.df)

def __getitem__(self, idx):

    row = self.df.iloc[idx]

    img_bgr = cv2.imread(row['path'])

    # ============================================================
    # SAFETY CHECK
    # ============================================================

    if img_bgr is None:

        img_bgr = np.zeros(
            (SZ, SZ, 3),
            dtype=np.uint8
        )

    img_rgb = cv2.cvtColor(
        img_bgr,
        cv2.COLOR_BGR2RGB
    )

    # ============================================================
    # AUGMENTATIONS
    # ============================================================

    if self.mode == 'train':

        if 'label' in row.index and row['label'] in MINORITY_CLASSES:

            tensor = minority_tf(img_rgb)

        else:

            tensor = train_tf(img_rgb)

    else:

        tensor = val_tf(img_rgb)

    # ============================================================
    # LABELED DATA
    # TRAIN / VAL
    # ============================================================

    if 'label_idx' in row.index:

        label = torch.tensor(
            int(row['label_idx']),
            dtype=torch.long
        )

        return tensor, label

    # ============================================================
    # UNLABELED TEST DATA
    # ============================================================

    else:

        return tensor, torch.tensor(-1)


def make_weighted_sampler(df):
    """Over-sample minority classes so each batch is roughly balanced."""
    counts  = df['label_idx'].value_counts().sort_index().values.astype(float)
    weights = 1.0 / counts
    sample_weights = torch.tensor(
        [weights[int(i)] for i in df['label_idx']], dtype=torch.float
    )
    return WeightedRandomSampler(
        weights     = sample_weights,
        num_samples = len(sample_weights),
        replacement = True
    )


def make_loaders(tr_df, va_df, te_df):
    tr_ds = WBCDataset(tr_df, mode='train')
    va_ds = WBCDataset(va_df, mode='val')
    te_ds = WBCDataset(te_df, mode='test')

    sampler = make_weighted_sampler(tr_df)

    tr_loader = DataLoader(tr_ds, batch_size=CFG['batch_size'],
                           sampler=sampler,
                           num_workers=CFG['num_workers'],
                           pin_memory=CFG['pin_memory'],
                           drop_last=True)
    va_loader = DataLoader(va_ds, batch_size=CFG['batch_size'],
                           shuffle=False,
                           num_workers=CFG['num_workers'],
                           pin_memory=CFG['pin_memory'])
    te_loader = DataLoader(te_ds, batch_size=CFG['batch_size'],
                           shuffle=False,
                           num_workers=CFG['num_workers'],
                           pin_memory=CFG['pin_memory'])
    return tr_loader, va_loader, te_loader


def get_class_weights(df):
    """Compute inverse-frequency class weights for CrossEntropyLoss."""
    y = df['label_idx'].values.astype(int)
    w = compute_class_weight('balanced', classes=np.unique(y), y=y)
    return torch.tensor(w, dtype=torch.float).to(device)


print("Dataset / sampler / loader helpers ready.")

## Cell 7 — Dataloader helper with WeightedRandomSampler

## Cell 8 — ConvNeXt-Tiny model

In [ ]:
# ── Cell 7 — Build ConvNeXt-Tiny model ───────────────────────────

def build_model(freeze_backbone=True):
    """
    ConvNeXt-Tiny pretrained on ImageNet.
    freeze_backbone=True : only the head trains for the first few epochs.
    At unfreeze_epoch the full network is unfrozen for fine-tuning.
    """
    model = timm.create_model(
        CFG['model_name'],
        pretrained  = True,
        num_classes = CFG['num_classes'],
    )
    if freeze_backbone:
        for name, param in model.named_parameters():
            if 'head' not in name:
                param.requires_grad = False

    model = model.to(device)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Trainable params : {trainable:,} / {total:,}")
    return model


def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"All layers unfrozen — {trainable:,} trainable params")


# Quick smoke-test
m = build_model(freeze_backbone=True)
print("Model built successfully.")
del m
torch.cuda.empty_cache()

## Cell 9 — Train epoch & evaluate functions

In [ ]:
# ── Cell 9 — Train epoch & evaluate functions ─────────────────────

# ── Mixup helper ─────────────────────────────────────────────────
def mixup_batch(imgs, labels, alpha=0.3):
    """
    Mixup only on majority classes (SNE, LY, MO, BL).
    Minority classes are left unchanged — too few samples to mix safely.
    """
    MAJORITY_IDXS = {CLASS2IDX[c] for c in ['SNE', 'LY', 'MO', 'BL']
                     if c in CLASS2IDX}
    majority_mask = torch.tensor(
        [l.item() in MAJORITY_IDXS for l in labels],
        device=labels.device
    )
    majority = majority_mask.nonzero(as_tuple=True)[0]

    if len(majority) < 2:
        return imgs, labels, labels, 1.0

    lam  = float(np.random.beta(alpha, alpha))
    lam  = max(lam, 1.0 - lam)

    perm      = majority[torch.randperm(len(majority), device=labels.device)]
    imgs_mix  = imgs.clone()
    labels_b  = labels.clone()

    imgs_mix[majority] = lam * imgs[majority] + (1.0 - lam) * imgs[perm]
    labels_b[majority] = labels[perm]

    return imgs_mix, labels, labels_b, lam


# ── Train one epoch ───────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, scaler,
                    use_mixup=False):
    model.train()
    total_loss = 0.0
    correct    = 0
    total      = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()

        if use_mixup and np.random.rand() < 0.5:
            imgs_m, lab_a, lab_b, lam = mixup_batch(imgs, labels, alpha=0.3)
            with autocast():
                logits = model(imgs_m)
                loss   = lam * criterion(logits, lab_a) + \
                         (1.0 - lam) * criterion(logits, lab_b)
        else:
            with autocast():
                logits = model(imgs)
                loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * len(labels)
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += len(labels)

    return total_loss / total, correct / total


# ── Evaluate (optional TTA) ───────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, criterion, n_tta=1):
    """
    n_tta=1  → standard eval (used inside run_training)
    n_tta=5  → TTA at test time (call manually after training)
    """
    model.eval()
    total_loss  = 0.0
    all_preds   = []
    all_targets = []

    for imgs, labels in loader:
        labels = labels.to(device)

        if n_tta == 1:
            imgs = imgs.to(device)
            with autocast():
                logits = model(imgs)
                loss   = criterion(logits, labels)
            total_loss += loss.item() * len(labels)
            all_preds  += logits.argmax(dim=1).cpu().tolist()

        else:
            batch_probs = torch.zeros(len(labels), CFG['num_classes'])
            # Pass 1 — original
            with autocast():
                logits      = model(imgs.to(device))
                loss        = criterion(logits, labels)
                batch_probs += torch.softmax(logits, dim=1).cpu()
            total_loss += loss.item() * len(labels)
            # Passes 2..n_tta — random flips/rotations
            for _ in range(n_tta - 1):
                imgs_aug = imgs.clone()
                if torch.rand(1).item() > 0.5:
                    imgs_aug = torch.flip(imgs_aug, dims=[3])
                if torch.rand(1).item() > 0.5:
                    imgs_aug = torch.flip(imgs_aug, dims=[2])
                k = torch.randint(0, 4, (1,)).item()
                imgs_aug = torch.rot90(imgs_aug, k, dims=[2, 3])
                with autocast():
                    logits_aug  = model(imgs_aug.to(device))
                batch_probs += torch.softmax(logits_aug, dim=1).cpu()
            batch_probs /= n_tta
            all_preds   += batch_probs.argmax(dim=1).tolist()

        all_targets += labels.cpu().tolist()

    n        = len(all_targets)
    acc      = sum(p == t for p, t in zip(all_preds, all_targets)) / n
    macro_f1 = f1_score(all_targets, all_preds,
                        average='macro', zero_division=0)
    return total_loss / n, acc, macro_f1, all_preds, all_targets


print("train_one_epoch() and evaluate() ready.")

## Cell 10 — Full training loop

In [ ]:
# ── Cell 10 — run_training() loop ─────────────────────────────────

def run_training(model, tr_loader, va_loader, criterion, fold_id='fold'):
    """
    Full training loop:
      - AMP mixed precision
      - Freeze → unfreeze backbone
      - Differential learning rates after unfreeze
      - Cosine LR scheduler
      - Mixup after unfreeze
      - Gradient clipping
      - Early stopping on macro-F1
    """
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CFG['lr'],
        weight_decay=CFG['weight_decay']
    )
    scaler    = GradScaler()
    best_f1   = 0.0
    no_improve = 0
    ckpt_path  = f"{CFG['save_dir']}/{fold_id}_best.pth"
    scheduler  = None

    history = {'tr_loss': [], 'va_loss': [],
               'tr_acc':  [], 'va_acc':  [], 'va_f1': []}

    for epoch in range(1, CFG['epochs'] + 1):

        # ── Unfreeze backbone ─────────────────────────────────────
        if epoch == CFG['unfreeze_epoch']:
            unfreeze_all(model)
            optimizer = torch.optim.AdamW([
                {'params': [p for n, p in model.named_parameters()
                            if 'head' not in n],
                 'lr': CFG['unfreeze_lr']},
                {'params': [p for n, p in model.named_parameters()
                            if 'head' in n],
                 'lr': CFG['lr']},
            ], weight_decay=CFG['weight_decay'])
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max   = CFG['epochs'] - CFG['unfreeze_epoch'] + 1,
                eta_min = 1e-7
            )
            print(f"  → Backbone unfrozen | "
                  f"backbone_lr={CFG['unfreeze_lr']:.0e} | "
                  f"head_lr={CFG['lr']:.0e}")

        use_mixup = (epoch >= CFG['unfreeze_epoch'])

        # ── Train ────────────────────────────────────────────────
        tr_loss, tr_acc = train_one_epoch(
            model, tr_loader, optimizer, criterion, scaler,
            use_mixup=use_mixup
        )

        # ── Validate ─────────────────────────────────────────────
        va_loss, va_acc, va_f1, _, _ = evaluate(model, va_loader, criterion)

        if scheduler is not None:
            scheduler.step()

        current_lr = optimizer.param_groups[0]['lr']

        history['tr_loss'].append(tr_loss)
        history['va_loss'].append(va_loss)
        history['tr_acc'].append(tr_acc)
        history['va_acc'].append(va_acc)
        history['va_f1'].append(va_f1)

        print(f"  Epoch {epoch:02d}/{CFG['epochs']}  "
              f"tr_loss={tr_loss:.4f}  va_loss={va_loss:.4f}  "
              f"tr_acc={tr_acc:.4f}  va_acc={va_acc:.4f}  "
              f"va_F1={va_f1:.4f}  lr={current_lr:.1e}  "
              f"mixup={use_mixup}")

        # ── Save best checkpoint ─────────────────────────────────
        if va_f1 > best_f1:
            best_f1 = va_f1
            torch.save(model.state_dict(), ckpt_path)
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= CFG['patience']:
                print(f"  Early stopping at epoch {epoch}")
                break

    print(f"\nBest val macro-F1 = {best_f1:.4f}")
    print(f"Checkpoint saved  → {ckpt_path}")
    return history, ckpt_path, best_f1


print("run_training() ready.")

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score
)

## Cell 11 — 3-Fold Cross Validation + OOF predictions
**What this does:** Splits train+val data into 3 folds. Each fold trains on 2 parts and predicts on the held-out part. OOF = out-of-fold predictions cover every training sample exactly once — giving an unbiased accuracy estimate without touching the test set.

**GPU time:** ~20-25 min on T4 (3 folds × 8 epochs × ~800 batches)


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 11 — 3-Fold CV + OOF (FIXED + FAST)
# Fix 1: make_loaders uses f_va not test_df → no CUDA assert
# Fix 2: OOF saved to disk after every fold → never lose progress
# Expected time: ~2 hrs total for 3 folds (images pre-cached)
# ══════════════════════════════════════════════════════════════════

import os
import pickle
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, balanced_accuracy_score,
    f1_score, precision_score, recall_score, accuracy_score
)

CLASSES = CFG['classes']
NC      = CFG['num_classes']

# Combine train + val for CV
trainval_df = pd.concat([train_df, val_df], ignore_index=True)
trainval_df = trainval_df.sample(
    frac=1, random_state=SEED
).reset_index(drop=True)

skf         = StratifiedKFold(
    n_splits=CFG['n_folds'], shuffle=True, random_state=SEED
)
oof_preds   = np.zeros(len(trainval_df), dtype=int)
oof_targets = trainval_df['label_idx'].values.astype(int)
fold_results = []

print(f"Starting {CFG['n_folds']}-fold CV on {len(trainval_df):,} images")
print(f"Images are pre-cached → training will be fast (~2 hrs total)")
print('=' * 60)

for fold, (tr_idx, va_idx) in enumerate(
        skf.split(trainval_df, trainval_df['label_idx']), start=1):

    print(f"\n--- FOLD {fold}/{CFG['n_folds']} ---")
    f_tr = trainval_df.iloc[tr_idx].reset_index(drop=True)
    f_va = trainval_df.iloc[va_idx].reset_index(drop=True)
    print(f"  train={len(f_tr):,}  val={len(f_va):,}")

    criterion = make_criterion(f_tr)

    # ── CRITICAL: f_va as 3rd arg — NEVER test_df ────────────────
    tr_loader, va_loader, _ = make_loaders(f_tr, f_va, f_va)

    model = build_model(freeze_backbone=True)

    hist, ckpt_path, best_f1 = run_training(
        model, tr_loader, va_loader, criterion,
        fold_id=f'fold{fold}'
    )

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    _, va_acc, va_f1, va_preds, va_tgt = evaluate(
        model, va_loader, criterion
    )
    oof_preds[va_idx] = va_preds

    fold_acc      = accuracy_score(va_tgt, va_preds)
    fold_ba       = balanced_accuracy_score(va_tgt, va_preds)
    fold_macro_f1 = f1_score(va_tgt, va_preds,
                              average='macro',    zero_division=0)
    fold_wt_f1    = f1_score(va_tgt, va_preds,
                              average='weighted', zero_division=0)
    fold_macro_p  = precision_score(va_tgt, va_preds,
                                     average='macro', zero_division=0)
    fold_macro_r  = recall_score(va_tgt, va_preds,
                                  average='macro', zero_division=0)
    fold_per_p    = precision_score(va_tgt, va_preds, average=None,
                                     zero_division=0,
                                     labels=list(range(NC)))
    fold_per_r    = recall_score(va_tgt, va_preds, average=None,
                                  zero_division=0,
                                  labels=list(range(NC)))
    fold_per_f1   = f1_score(va_tgt, va_preds, average=None,
                              zero_division=0, labels=list(range(NC)))

    fold_results.append({
        'fold'    : fold,
        'acc'     : fold_acc,
        'ba'      : fold_ba,
        'macro_f1': fold_macro_f1,
        'wt_f1'   : fold_wt_f1,
        'macro_p' : fold_macro_p,
        'macro_r' : fold_macro_r,
        'per_p'   : fold_per_p,
        'per_r'   : fold_per_r,
        'per_f1'  : fold_per_f1,
        'preds'   : va_preds,
        'targets' : va_tgt,
        'history' : hist,
    })

    print(f"\n  {'='*52}")
    print(f"  FOLD {fold} RESULTS")
    print(f"  {'='*52}")
    print(f"  Accuracy          : {fold_acc:.4f}")
    print(f"  Balanced Accuracy : {fold_ba:.4f}")
    print(f"  Macro Precision   : {fold_macro_p:.4f}")
    print(f"  Macro Recall      : {fold_macro_r:.4f}")
    print(f"  Macro F1          : {fold_macro_f1:.4f}")
    print(f"  Weighted F1       : {fold_wt_f1:.4f}")
    print(f"\n  Per-class breakdown:")
    print(f"  {'Class':<8} {'Precision':>10} {'Recall':>10} {'F1':>10}")
    print(f"  {'-'*42}")
    for i, cls in enumerate(CLASSES):
        print(f"  {cls:<8} {fold_per_p[i]:>10.4f} "
              f"{fold_per_r[i]:>10.4f} {fold_per_f1[i]:>10.4f}")

    # Confusion matrix
    cm      = confusion_matrix(va_tgt, va_preds, labels=list(range(NC)))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(22, 9))
    fig.suptitle(
        f"Fold {fold} — Confusion Matrix\n"
        f"Acc={fold_acc:.4f}  Macro-F1={fold_macro_f1:.4f}  "
        f"Bal-Acc={fold_ba:.4f}",
        fontsize=13, fontweight='bold'
    )
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES,
                linewidths=0.3, ax=axes[0], cbar=True,
                annot_kws={'size': 7})
    axes[0].set_title(f'Fold {fold} — Raw Counts', fontsize=11)
    axes[0].set_xlabel('Predicted Class', fontsize=10)
    axes[0].set_ylabel('True Class',      fontsize=10)
    axes[0].tick_params(axis='x', rotation=45, labelsize=8)
    axes[0].tick_params(axis='y', rotation=0,  labelsize=8)

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES,
                linewidths=0.3, ax=axes[1], cbar=True,
                vmin=0, vmax=1, annot_kws={'size': 7})
    axes[1].set_title(f'Fold {fold} — Normalised', fontsize=11)
    axes[1].set_xlabel('Predicted Class', fontsize=10)
    axes[1].set_ylabel('True Class',      fontsize=10)
    axes[1].tick_params(axis='x', rotation=45, labelsize=8)
    axes[1].tick_params(axis='y', rotation=0,  labelsize=8)

    plt.tight_layout()
    fold_cm_path = os.path.join(
        CFG['save_dir'], f'fold{fold}_confusion_matrix.png'
    )
    plt.savefig(fold_cm_path, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f"  Confusion matrix saved → {fold_cm_path}")

    # ── Save after every fold — never lose progress ───────────────
    np.save('/kaggle/working/oof_preds.npy',   oof_preds)
    np.save('/kaggle/working/oof_targets.npy', oof_targets)
    with open('/kaggle/working/fold_results.pkl', 'wb') as f:
        pickle.dump(fold_results, f)
    print(f"  OOF saved after fold {fold} ✓")

    del model, tr_loader, va_loader
    torch.cuda.empty_cache()


# ══════════════════════════════════════════════════════════════════
# OOF Summary
# ══════════════════════════════════════════════════════════════════
oof_f1      = f1_score(oof_targets, oof_preds,
                        average='macro',    zero_division=0)
oof_wt_f1   = f1_score(oof_targets, oof_preds,
                        average='weighted', zero_division=0)
oof_ba      = balanced_accuracy_score(oof_targets, oof_preds)
oof_acc     = accuracy_score(oof_targets, oof_preds)
oof_macro_p = precision_score(oof_targets, oof_preds,
                               average='macro', zero_division=0)
oof_macro_r = recall_score(oof_targets, oof_preds,
                            average='macro', zero_division=0)
oof_per_p   = precision_score(oof_targets, oof_preds, average=None,
                               zero_division=0, labels=list(range(NC)))
oof_per_r   = recall_score(oof_targets, oof_preds, average=None,
                            zero_division=0, labels=list(range(NC)))
oof_per_f1  = f1_score(oof_targets, oof_preds, average=None,
                        zero_division=0, labels=list(range(NC)))

print("\n" + "=" * 60)
print("  CROSS-VALIDATION COMPLETE — PER FOLD SUMMARY")
print("=" * 60)
print(f"  {'Fold':<6} {'Acc':>8} {'Bal-Acc':>9} {'Mac-P':>8} "
      f"{'Mac-R':>8} {'Mac-F1':>8} {'Wt-F1':>8}")
print(f"  {'-'*57}")
for r in fold_results:
    print(f"  {r['fold']:<6} {r['acc']:>8.4f} {r['ba']:>9.4f} "
          f"{r['macro_p']:>8.4f} {r['macro_r']:>8.4f} "
          f"{r['macro_f1']:>8.4f} {r['wt_f1']:>8.4f}")

mean_acc = np.mean([r['acc']      for r in fold_results])
mean_ba  = np.mean([r['ba']       for r in fold_results])
mean_p   = np.mean([r['macro_p']  for r in fold_results])
mean_r   = np.mean([r['macro_r']  for r in fold_results])
mean_f1  = np.mean([r['macro_f1'] for r in fold_results])
mean_wf1 = np.mean([r['wt_f1']   for r in fold_results])

print(f"  {'-'*57}")
print(f"  {'Mean':<6} {mean_acc:>8.4f} {mean_ba:>9.4f} "
      f"{mean_p:>8.4f} {mean_r:>8.4f} {mean_f1:>8.4f} {mean_wf1:>8.4f}")

print(f"\n  Per-class OOF metrics:")
print(f"  {'Class':<8} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print(f"  {'-'*42}")
for i, cls in enumerate(CLASSES):
    print(f"  {cls:<8} {oof_per_p[i]:>10.4f} "
          f"{oof_per_r[i]:>10.4f} {oof_per_f1[i]:>10.4f}")

print(f"\n  OOF (all folds combined):")
print(f"  Accuracy          : {oof_acc:.4f}")
print(f"  Balanced Accuracy : {oof_ba:.4f}")
print(f"  Macro Precision   : {oof_macro_p:.4f}")
print(f"  Macro Recall      : {oof_macro_r:.4f}")
print(f"  Macro F1          : {oof_f1:.4f}")
print(f"  Weighted F1       : {oof_wt_f1:.4f}")

# OOF confusion matrix
cm_oof      = confusion_matrix(oof_targets, oof_preds,
                                labels=list(range(NC)))
cm_oof_norm = cm_oof.astype(float) / cm_oof.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
fig.suptitle(
    f"OOF Confusion Matrix (All {CFG['n_folds']} Folds Combined)\n"
    f"Acc={oof_acc:.4f}  Macro-F1={oof_f1:.4f}  "
    f"Bal-Acc={oof_ba:.4f}",
    fontsize=13, fontweight='bold'
)
sns.heatmap(cm_oof, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES,
            linewidths=0.3, ax=axes[0], cbar=True,
            annot_kws={'size': 7})
axes[0].set_title('OOF — Raw Counts', fontsize=11)
axes[0].set_xlabel('Predicted Class', fontsize=10)
axes[0].set_ylabel('True Class',      fontsize=10)
axes[0].tick_params(axis='x', rotation=45, labelsize=8)
axes[0].tick_params(axis='y', rotation=0,  labelsize=8)

sns.heatmap(cm_oof_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES,
            linewidths=0.3, ax=axes[1], cbar=True,
            vmin=0, vmax=1, annot_kws={'size': 7})
axes[1].set_title('OOF — Normalised', fontsize=11)
axes[1].set_xlabel('Predicted Class', fontsize=10)
axes[1].set_ylabel('True Class',      fontsize=10)
axes[1].tick_params(axis='x', rotation=45, labelsize=8)
axes[1].tick_params(axis='y', rotation=0,  labelsize=8)

plt.tight_layout()
oof_cm_path = os.path.join(
    CFG['save_dir'], 'oof_confusion_matrix_all_folds.png'
)
plt.savefig(oof_cm_path, dpi=120, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f"\nOOF confusion matrix saved → {oof_cm_path}")
print("=" * 60)

## Cell 12 — Train final model on full train+val, evaluate on test set

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 12 — Final model: retrain on trainval, evaluate on val
# FIXED: test_df never used as val — no CUDA assert
# FIXED: trainval_df reuse check — not redefined if already exists
# ══════════════════════════════════════════════════════════════════

import os
import pickle
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, balanced_accuracy_score,
    f1_score, precision_score, recall_score, accuracy_score
)
from torch.utils.data import DataLoader

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

CLASSES = CFG['classes']
NC      = CFG['num_classes']

# ── Verify CUDA ───────────────────────────────────────────────────
try:
    _ = torch.zeros(1).cuda()
    print("CUDA OK")
except Exception as e:
    print(f"CUDA ERROR: {e}")
    raise SystemExit("Restart kernel and rerun all cells")

torch.cuda.empty_cache()

# ── trainval_df — reuse from Cell 11 if available ────────────────
if 'trainval_df' not in dir() or len(trainval_df) == 0:
    trainval_df = pd.concat([train_df, val_df], ignore_index=True)
    trainval_df = trainval_df.sample(
        frac=1, random_state=SEED
    ).reset_index(drop=True)
    print("trainval_df rebuilt from train + val")
else:
    print("trainval_df reused from Cell 11")

print(f"\ntrainval : {len(trainval_df):,} rows")
print(f"val      : {len(val_df):,} rows  (early stopping)")
print(f"test     : {len(test_df):,} rows  (inference only)")

# ── Criterion ─────────────────────────────────────────────────────
final_criterion = make_criterion(trainval_df)

# ── Loaders ──────────────────────────────────────────────────────
# val_df as 2nd AND 3rd arg — NEVER test_df
# test_df has label_idx=-1 → CUDA assert in loss if used here
final_tr_loader, final_va_loader, _ = make_loaders(
    trainval_df, val_df, val_df
)

# ── Build + train ────────────────────────────────────────────────
final_model = build_model(freeze_backbone=True)

final_hist, final_ckpt, final_best_f1 = run_training(
    final_model, final_tr_loader, final_va_loader,
    final_criterion, fold_id='final'
)

# ── Load best checkpoint ──────────────────────────────────────────
final_model.load_state_dict(
    torch.load(final_ckpt, map_location=device)
)
final_model.eval()
print(f"\nBest checkpoint : {final_ckpt}")
print(f"Best val F1     : {final_best_f1:.4f}")

# ── Save OOF immediately after training completes ────────────────
try:
    np.save('/kaggle/working/oof_preds.npy',   oof_preds)
    np.save('/kaggle/working/oof_targets.npy', oof_targets)
    with open('/kaggle/working/fold_results.pkl', 'wb') as f:
        pickle.dump(fold_results, f)
    print("OOF saved to /kaggle/working/ ✓")
except Exception as e:
    print(f"OOF save warning: {e}")

# ══════════════════════════════════════════════════════════════════
# Evaluate on val_df (real labels → proper metrics)
# ══════════════════════════════════════════════════════════════════
print("\nEvaluating on val set (TTA n=5)...")
_, va_acc, va_f1, va_preds, va_targets = evaluate(
    final_model, final_va_loader, final_criterion, n_tta=5
)
va_ba      = balanced_accuracy_score(va_targets, va_preds)
va_macro_p = precision_score(va_targets, va_preds,
                              average='macro', zero_division=0)
va_macro_r = recall_score(va_targets, va_preds,
                           average='macro', zero_division=0)
va_per_f1  = f1_score(va_targets, va_preds,
                       average=None, zero_division=0,
                       labels=list(range(NC)))

print(f"\nVAL SET RESULTS")
print(f"  Accuracy          : {va_acc:.4f}")
print(f"  Balanced Accuracy : {va_ba:.4f}")
print(f"  Macro Precision   : {va_macro_p:.4f}")
print(f"  Macro Recall      : {va_macro_r:.4f}")
print(f"  Macro F1          : {va_f1:.4f}")
print(f"\n  Per-class F1:")
print(f"  {'Class':<8} {'F1':>8}")
print(f"  {'-'*18}")
for i, cls in enumerate(CLASSES):
    flag = ' ← minority' if cls in MINORITY_CLASSES else ''
    print(f"  {cls:<8} {va_per_f1[i]:>8.4f}{flag}")

# ══════════════════════════════════════════════════════════════════
# Test inference — labels completely ignored
# ══════════════════════════════════════════════════════════════════
print("\nRunning test inference (no labels)...")

test_dataset = WBCDataset(test_df, mode='test')
test_loader  = DataLoader(
    test_dataset,
    batch_size  = CFG['batch_size'],
    shuffle     = False,
    num_workers = CFG['num_workers'],
    pin_memory  = CFG['pin_memory'],
)

all_test_preds = []
all_test_probs = []

final_model.eval()
with torch.no_grad():
    for imgs, _ in test_loader:   # _ = label ignored completely
        imgs   = imgs.to(device)
        with torch.cuda.amp.autocast():
            logits = final_model(imgs)
        probs  = torch.softmax(logits, dim=1)
        preds  = probs.argmax(dim=1)
        all_test_preds.append(preds.cpu().numpy())
        all_test_probs.append(probs.cpu().numpy())

all_test_preds = np.concatenate(all_test_preds)
all_test_probs = np.concatenate(all_test_probs)

print(f"Test inference complete: {len(all_test_preds):,} predictions")
print("\nPrediction distribution:")
pred_dist = pd.Series(all_test_preds).map(IDX2CLASS).value_counts()
for cls, count in pred_dist.items():
    bar = "█" * int(count / pred_dist.max() * 20)
    print(f"  {cls:<6} {count:>6,}  {bar}")

# Save predictions CSV
test_pred_df = test_df[['ID']].copy()
test_pred_df['predicted_label']     = [IDX2CLASS[p] for p in all_test_preds]
test_pred_df['predicted_label_idx'] = all_test_preds
for i, cls in enumerate(CLASSES):
    test_pred_df[f'prob_{cls}'] = all_test_probs[:, i]

pred_path = os.path.join(CFG['save_dir'], 'test_predictions.csv')
test_pred_df.to_csv(pred_path, index=False)
print(f"Test predictions saved → {pred_path}")

# ══════════════════════════════════════════════════════════════════
# Val confusion matrix
# ══════════════════════════════════════════════════════════════════
cm_va      = confusion_matrix(va_targets, va_preds,
                               labels=list(range(NC)))
cm_va_norm = cm_va.astype(float) / cm_va.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
fig.suptitle(
    f"Final Model — Val Confusion Matrix\n"
    f"Acc={va_acc:.4f}  Macro-F1={va_f1:.4f}  "
    f"Bal-Acc={va_ba:.4f}",
    fontsize=13, fontweight='bold'
)
sns.heatmap(cm_va, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASSES, yticklabels=CLASSES,
            linewidths=0.3, ax=axes[0], cbar=True,
            annot_kws={'size': 7})
axes[0].set_title('Val — Raw Counts',  fontsize=11)
axes[0].set_xlabel('Predicted Class',  fontsize=10)
axes[0].set_ylabel('True Class',       fontsize=10)
axes[0].tick_params(axis='x', rotation=45, labelsize=8)
axes[0].tick_params(axis='y', rotation=0,  labelsize=8)

sns.heatmap(cm_va_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=CLASSES, yticklabels=CLASSES,
            linewidths=0.3, ax=axes[1], cbar=True,
            vmin=0, vmax=1, annot_kws={'size': 7})
axes[1].set_title('Val — Normalised',  fontsize=11)
axes[1].set_xlabel('Predicted Class',  fontsize=10)
axes[1].set_ylabel('True Class',       fontsize=10)
axes[1].tick_params(axis='x', rotation=45, labelsize=8)
axes[1].tick_params(axis='y', rotation=0,  labelsize=8)

plt.tight_layout()
cm_path = os.path.join(CFG['save_dir'], 'final_val_confusion_matrix.png')
plt.savefig(cm_path, dpi=120, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f"Confusion matrix saved → {cm_path}")

# ══════════════════════════════════════════════════════════════════
# Training curves
# ══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Final Model — Training Curves',
             fontsize=13, fontweight='bold')

axes[0].plot(final_hist['tr_loss'], label='Train', color='steelblue')
axes[0].plot(final_hist['va_loss'], label='Val',   color='orange')
axes[0].set_title('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(final_hist['tr_acc'], label='Train', color='steelblue')
axes[1].plot(final_hist['va_acc'], label='Val',   color='orange')
axes[1].set_title('Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(final_hist['va_f1'], label='Val Macro-F1', color='green')
axes[2].set_title('Val Macro-F1')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
curves_path = os.path.join(CFG['save_dir'], 'final_training_curves.png')
plt.savefig(curves_path, dpi=120, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f"Training curves saved → {curves_path}")

print("\n" + "="*60)
print("Cell 12 complete.")
print(f"  Val Macro-F1     : {va_f1:.4f}")
print(f"  Val Bal-Acc      : {va_ba:.4f}")
print(f"  Test predictions : {len(all_test_preds):,} saved")
print("="*60)

In [ ]:
# ── Cell 12 — Final model: retrain on full trainval, evaluate on test ─

print("Training final model on full trainval set ...")

final_criterion  = make_criterion(trainval_df)
final_tr_loader, _, final_te_loader = make_loaders(
    trainval_df, test_df, test_df
)
final_model = build_model(freeze_backbone=True)

final_hist, final_ckpt, final_best_f1 = run_training(
    final_model, final_tr_loader, final_te_loader,
    final_criterion, fold_id='final'
)

# Load best checkpoint
final_model.load_state_dict(
    torch.load(final_ckpt, map_location=device)
)

# Evaluate on test set (with TTA)
print("\nEvaluating on held-out test set (TTA n=5) ...")
_, te_acc, te_f1, te_preds, te_targets = evaluate(
    final_model, final_te_loader, final_criterion, n_tta=5
)

te_ba      = balanced_accuracy_score(te_targets, te_preds)
te_macro_p = precision_score(te_targets, te_preds,
                             average='macro', zero_division=0)
te_macro_r = recall_score(te_targets,    te_preds,
                          average='macro', zero_division=0)
te_per     = f1_score(te_targets, te_preds,
                      average=None, zero_division=0, labels=list(range(NC)))

print(f"\nTEST SET RESULTS")
print(f"  Accuracy          : {te_acc:.4f}")
print(f"  Balanced Accuracy : {te_ba:.4f}")
print(f"  Macro Precision   : {te_macro_p:.4f}")
print(f"  Macro Recall      : {te_macro_r:.4f}")
print(f"  Macro F1          : {te_f1:.4f}")
print(f"\n  Per-class F1:")
for i, cls in enumerate(CLASSES):
    print(f"    {cls:<8} {te_per[i]:.4f}")

# Test confusion matrix
cm_te      = confusion_matrix(te_targets, te_preds, labels=list(range(NC)))
cm_te_norm = cm_te.astype(float) / cm_te.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
fig.suptitle(
    f"Test Set Confusion Matrix\n"
    f"Acc={te_acc:.4f}  Macro-F1={te_f1:.4f}  Bal-Acc={te_ba:.4f}",
    fontsize=13, fontweight='bold'
)
sns.heatmap(cm_te, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASSES, yticklabels=CLASSES,
            linewidths=0.3, ax=axes[0], cbar=True, annot_kws={'size': 7})
axes[0].set_title('Test — Raw Counts', fontsize=11)
axes[0].set_xlabel('Predicted Class', fontsize=10)
axes[0].set_ylabel('True Class',      fontsize=10)
axes[0].tick_params(axis='x', rotation=45, labelsize=8)
axes[0].tick_params(axis='y', rotation=0,  labelsize=8)

sns.heatmap(cm_te_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=CLASSES, yticklabels=CLASSES,
            linewidths=0.3, ax=axes[1], cbar=True,
            vmin=0, vmax=1, annot_kws={'size': 7})
axes[1].set_title('Test — Normalised', fontsize=11)
axes[1].set_xlabel('Predicted Class', fontsize=10)
axes[1].set_ylabel('True Class',      fontsize=10)
axes[1].tick_params(axis='x', rotation=45, labelsize=8)
axes[1].tick_params(axis='y', rotation=0,  labelsize=8)

plt.tight_layout()
te_cm_path = os.path.join(CFG['save_dir'], 'test_confusion_matrix.png')
plt.savefig(te_cm_path, dpi=120, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f"Test confusion matrix saved → {te_cm_path}")

# Training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Final Model — Training Curves', fontsize=13, fontweight='bold')

axes[0].plot(final_hist['tr_loss'], label='Train')
axes[0].plot(final_hist['va_loss'], label='Val')
axes[0].set_title('Loss');  axes[0].legend()

axes[1].plot(final_hist['tr_acc'], label='Train')
axes[1].plot(final_hist['va_acc'], label='Val')
axes[1].set_title('Accuracy');  axes[1].legend()

axes[2].plot(final_hist['va_f1'], label='Val Macro-F1', color='green')
axes[2].set_title('Val Macro-F1');  axes[2].legend()

plt.tight_layout()
curves_path = os.path.join(CFG['save_dir'], 'training_curves.png')
plt.savefig(curves_path, dpi=120, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f"Training curves saved → {curves_path}")

In [ ]:
# CELL A — Kill only the corrupted CUDA context, keep all dataframes
import torch
import gc

# Force kill GPU context
torch.cuda.synchronize = lambda: None   # disable sync calls
torch.cuda._initialized = False

gc.collect()
torch.cuda.empty_cache = lambda: None   # disable cache clear calls

print("CUDA calls neutralized")
print("train_df:", len(train_df))
print("val_df  :", len(val_df))
print("test_df :", len(test_df))
print("fold_results available:", len(fold_results))

In [ ]:
# CELL B — Reinitialize CUDA cleanly
import torch
import importlib
importlib.reload(torch.cuda)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

# Test CUDA is working
try:
    x = torch.zeros(3, 3).to(device)
    print("CUDA working:", x.device)
except Exception as e:
    print("CUDA still broken:", e)
    print("Only option now: Factory Reset → Session → Restart")

In [ ]:
import os
files = sorted(os.listdir('/kaggle/working/checkpoints'))
print('\n'.join(files))

In [ ]:
import pickle, numpy as np

# Save everything that took 4 hours to compute
np.save('/kaggle/working/oof_preds.npy',    oof_preds)
np.save('/kaggle/working/oof_targets.npy',  oof_targets)

with open('/kaggle/working/fold_results.pkl', 'wb') as f:
    pickle.dump(fold_results, f)

print("Saved:")
print("  oof_preds    :", oof_preds.shape)
print("  oof_targets  :", oof_targets.shape)
print("  fold_results :", len(fold_results), "folds")
print("\nNow safe to restart kernel.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 12 — FINAL MODEL TRAINING + VALIDATION + TEST PREDICTION
# FIXED VERSION FOR UNLABELED KAGGLE TEST SET
# ══════════════════════════════════════════════════════════════════

print("=" * 60)
print("TRAINING FINAL MODEL ON FULL TRAIN+VAL")
print("=" * 60)

# ============================================================
# FINAL CRITERION
# ============================================================

final_criterion = make_criterion(trainval_df)

# ============================================================
# IMPORTANT FIX
# TEST SET IS UNLABELED
# SO:
#   val_df  → evaluation
#   test_df → prediction only
# ============================================================

final_tr_loader, final_va_loader, final_te_loader = make_loaders(
    trainval_df,
    val_df,
    test_df
)

# ============================================================
# BUILD MODEL
# ============================================================

final_model = build_model(
    freeze_backbone=True
)

# ============================================================
# TRAIN
# ============================================================

final_hist, final_ckpt, final_best_f1 = run_training(
    final_model,
    final_tr_loader,
    final_va_loader,
    final_criterion,
    fold_id='final'
)

# ============================================================
# LOAD BEST CHECKPOINT
# ============================================================

print("\nLoading best checkpoint...\n")

final_model.load_state_dict(
    torch.load(
        final_ckpt,
        map_location=device
    )
)

# ============================================================
# VALIDATION EVALUATION
# ============================================================

print("\n" + "=" * 60)
print("FINAL VALIDATION EVALUATION")
print("=" * 60)

va_loss, va_acc, va_f1, va_preds, va_targets = evaluate(
    final_model,
    final_va_loader,
    final_criterion,
    n_tta=5
)

# ============================================================
# METRICS
# ============================================================

va_ba = balanced_accuracy_score(
    va_targets,
    va_preds
)

va_macro_p = precision_score(
    va_targets,
    va_preds,
    average='macro',
    zero_division=0
)

va_macro_r = recall_score(
    va_targets,
    va_preds,
    average='macro',
    zero_division=0
)

va_per = f1_score(
    va_targets,
    va_preds,
    average=None,
    zero_division=0,
    labels=list(range(NC))
)

# ============================================================
# PRINT RESULTS
# ============================================================

print(f"\nFINAL VALIDATION RESULTS\n")

print(f"Accuracy          : {va_acc:.4f}")
print(f"Balanced Accuracy : {va_ba:.4f}")
print(f"Macro Precision   : {va_macro_p:.4f}")
print(f"Macro Recall      : {va_macro_r:.4f}")
print(f"Macro F1          : {va_f1:.4f}")

print(f"\nPer-class F1:\n")

for i, cls in enumerate(CLASSES):

    print(f"{cls:<8} : {va_per[i]:.4f}")

# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    va_targets,
    va_preds,
    labels=list(range(NC))
)

cm_norm = cm.astype(float) / (
    cm.sum(axis=1, keepdims=True) + 1e-8
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(22,9)
)

fig.suptitle(
    f"Validation Confusion Matrix\n"
    f"Acc={va_acc:.4f} | "
    f"MacroF1={va_f1:.4f} | "
    f"BalAcc={va_ba:.4f}",
    fontsize=13,
    fontweight='bold'
)

# ============================================================
# RAW CM
# ============================================================

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=CLASSES,
    yticklabels=CLASSES,
    linewidths=0.3,
    ax=axes[0],
    cbar=True,
    annot_kws={'size':7}
)

axes[0].set_title(
    'Validation — Raw Counts',
    fontsize=11
)

axes[0].set_xlabel(
    'Predicted',
    fontsize=10
)

axes[0].set_ylabel(
    'True',
    fontsize=10
)

axes[0].tick_params(
    axis='x',
    rotation=45,
    labelsize=8
)

axes[0].tick_params(
    axis='y',
    rotation=0,
    labelsize=8
)

# ============================================================
# NORMALIZED CM
# ============================================================

sns.heatmap(
    cm_norm,
    annot=True,
    fmt='.2f',
    cmap='Greens',
    xticklabels=CLASSES,
    yticklabels=CLASSES,
    linewidths=0.3,
    ax=axes[1],
    cbar=True,
    vmin=0,
    vmax=1,
    annot_kws={'size':7}
)

axes[1].set_title(
    'Validation — Normalized',
    fontsize=11
)

axes[1].set_xlabel(
    'Predicted',
    fontsize=10
)

axes[1].set_ylabel(
    'True',
    fontsize=10
)

axes[1].tick_params(
    axis='x',
    rotation=45,
    labelsize=8
)

axes[1].tick_params(
    axis='y',
    rotation=0,
    labelsize=8
)

plt.tight_layout()

cm_path = os.path.join(
    CFG['save_dir'],
    'final_validation_confusion_matrix.png'
)

plt.savefig(
    cm_path,
    dpi=120,
    bbox_inches='tight'
)

plt.close(fig)

print(f"\nConfusion matrix saved:")
print(cm_path)

# ============================================================
# TRAINING CURVES
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18,5)
)

fig.suptitle(
    'Final Model Training Curves',
    fontsize=13,
    fontweight='bold'
)

# LOSS
axes[0].plot(
    final_hist['tr_loss'],
    label='Train'
)

axes[0].plot(
    final_hist['va_loss'],
    label='Val'
)

axes[0].set_title('Loss')
axes[0].legend()

# ACC
axes[1].plot(
    final_hist['tr_acc'],
    label='Train'
)

axes[1].plot(
    final_hist['va_acc'],
    label='Val'
)

axes[1].set_title('Accuracy')
axes[1].legend()

# F1
axes[2].plot(
    final_hist['va_f1'],
    label='Val Macro-F1'
)

axes[2].set_title('Macro-F1')
axes[2].legend()

plt.tight_layout()

curves_path = os.path.join(
    CFG['save_dir'],
    'final_training_curves.png'
)

plt.savefig(
    curves_path,
    dpi=120,
    bbox_inches='tight'
)

plt.close(fig)

print(f"\nTraining curves saved:")
print(curves_path)

# ============================================================
# TEST PREDICTION
# IMPORTANT:
# TEST SET IS UNLABELED
# ============================================================

print("\n" + "=" * 60)
print("GENERATING TEST PREDICTIONS")
print("=" * 60)

final_model.eval()

test_preds = []

with torch.no_grad():

    for imgs, _ in final_te_loader:

        imgs = imgs.to(device)

        logits = final_model(imgs)

        preds = torch.argmax(
            logits,
            dim=1
        )

        test_preds.extend(
            preds.cpu().numpy()
        )

# ============================================================
# CONVERT IDX → CLASS NAME
# ============================================================

test_pred_labels = [
    IDX2CLASS[p]
    for p in test_preds
]

# ============================================================
# SUBMISSION FILE
# ============================================================

submission_df = pd.DataFrame({

    'ID': test_df['ID'],
    'prediction': test_pred_labels
})

submission_path = os.path.join(
    CFG['save_dir'],
    'submission.csv'
)

submission_df.to_csv(
    submission_path,
    index=False
)

print("\nSubmission saved:")
print(submission_path)

print("\nSample predictions:\n")

print(submission_df.head())

print("\nFINAL PIPELINE COMPLETED SUCCESSFULLY")

In [ ]:
# ── Cell 13 — Save model weights + results ────────────────────────

import pickle, os

# Save final model weights
torch.save(
    final_model.state_dict(),
    f"{CFG['save_dir']}/convnext_final.pth"
)

# Compile results dict
# NOTE: Cell 12 evaluates on val_df (not test_df), so we use va_* variables.
# test_df has no real labels — only predictions are saved (test_predictions.csv).
results = {
    'model'          : 'ConvNeXt-Tiny',
    'val_f1'         : float(va_f1),
    'val_acc'        : float(va_acc),
    'val_ba'         : float(va_ba),
    'val_macro_p'    : float(va_macro_p),
    'val_macro_r'    : float(va_macro_r),
    'oof_f1'         : float(oof_f1),
    'oof_ba'         : float(oof_ba),
    'oof_per_class'  : {c: float(oof_per_f1[i]) for i, c in enumerate(CLASSES)},
    'val_per_class'  : {c: float(va_per_f1[i])  for i, c in enumerate(CLASSES)},
    'fold_results'   : fold_results,
    'mean_cv_f1'     : float(mean_f1),
    'mean_cv_acc'    : float(mean_acc),
    'test_pred_path' : os.path.join(CFG['save_dir'], 'test_predictions.csv'),
}

with open(f"{CFG['save_dir']}/convnext_results.pkl", 'wb') as f:
    pickle.dump(results, f)

# Check all expected output files
print("Outputs saved to /kaggle/working/checkpoints/:")
for fname in [
    'convnext_final.pth',
    'convnext_results.pkl',
    'final_training_curves.png',
    'final_val_confusion_matrix.png',
    'oof_confusion_matrix_all_folds.png',
    'test_predictions.csv',
]:
    fpath = os.path.join(CFG['save_dir'], fname)
    if os.path.exists(fpath):
        kb = os.path.getsize(fpath) / 1024
        print(f"  ✓ {fname}  ({kb:.0f} KB)")
    else:
        print(f"  ✗ {fname}  (NOT FOUND)")

print(f"\n{'='*50}")
print("  13-CLASS WBC PIPELINE COMPLETE")
print(f"{'='*50}")
print(f"  Classes      : {CFG['classes']}")
print(f"  Val Macro-F1 : {float(va_f1):.4f}")
print(f"  Val Bal-Acc  : {float(va_ba):.4f}")
print(f"  OOF  F1      : {float(oof_f1):.4f}")
print(f"  Mean CV F1   : {float(mean_f1):.4f}")
print(f"  Test preds   : {len(all_test_preds):,} rows → test_predictions.csv")
print("Session can now disconnect safely.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# IMPROVEMENT 2 — Per-Class Threshold Calibration (v2)
# Fix: test_df has label_idx=-1 (no ground truth) →
#      skip all metric computation on test set,
#      only run inference + save predictions.
#      All calibration/metrics use val_df only.
# ══════════════════════════════════════════════════════════════════

import numpy as np
import pickle, os
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast
from sklearn.metrics import (
    f1_score, balanced_accuracy_score,
    accuracy_score, classification_report
)

CLASSES = CFG['classes']
NC      = CFG['num_classes']

print("=" * 60)
print("THRESHOLD CALIBRATION (v2 — val-only metrics)")
print("=" * 60)

# ════════════════════════════════════════════════════════════════
# Step 1 — Collect logits on val_df (has real labels)
# ════════════════════════════════════════════════════════════════
final_model.eval()

va_loader_cal = DataLoader(
    WBCDataset(val_df, mode='val'),
    batch_size  = CFG['batch_size'],
    shuffle     = False,
    num_workers = CFG['num_workers'],
    pin_memory  = CFG['pin_memory'],
)

va_logits_list, va_true_list = [], []
with torch.no_grad():
    for imgs, labels in va_loader_cal:
        imgs = imgs.to(device)
        with autocast():
            logits = final_model(imgs)
        va_logits_list.append(logits.cpu().float())
        va_true_list.extend(labels.numpy())

va_logits = torch.cat(va_logits_list, dim=0)   # (N_val, NC)
va_true   = np.array(va_true_list)              # (N_val,) all in 0..12

# ════════════════════════════════════════════════════════════════
# Step 2 — Temperature scaling on val_df
# ════════════════════════════════════════════════════════════════
print("\n[1/4] Temperature scaling ...")

best_T    = 1.0
best_T_f1 = 0.0

for T in np.arange(0.5, 3.05, 0.05):
    probs_T = F.softmax(va_logits / T, dim=1).numpy()
    preds_T = probs_T.argmax(axis=1)
    f1_T    = f1_score(va_true, preds_T, average='macro', zero_division=0)
    if f1_T > best_T_f1:
        best_T_f1 = f1_T
        best_T    = float(T)

va_probs      = F.softmax(va_logits / best_T, dim=1).numpy()
argmax_preds  = va_probs.argmax(axis=1)
baseline_f1   = f1_score(va_true, argmax_preds, average='macro', zero_division=0)
print(f"  Best T={best_T:.2f}  val macro-F1={best_T_f1:.4f}  "
      f"(raw argmax baseline={baseline_f1:.4f})")

# ════════════════════════════════════════════════════════════════
# Step 3 — Vectorized per-class threshold search on val_df
# ════════════════════════════════════════════════════════════════
print("\n[2/4] Grid-searching per-class thresholds on val_df ...")

def apply_thresholds_vec(probs, thresholds):
    """Vectorized threshold application. Always returns a valid prediction."""
    top_class = probs.argmax(axis=1)
    top_prob  = probs[np.arange(len(probs)), top_class]
    fails     = top_prob < thresholds[top_class]
    preds     = top_class.copy()
    if fails.any():
        fail_probs = probs[fails]                           # (F, NC)
        thresh_mat = thresholds[np.newaxis, :]              # (1, NC)
        masked     = np.where(fail_probs >= thresh_mat,
                              fail_probs, -1.0)
        best_alt   = masked.argmax(axis=1)
        all_fail   = masked.max(axis=1) < 0                 # no class meets threshold
        best_alt[all_fail] = fail_probs[all_fail].argmax(axis=1)  # safe fallback
        preds[fails] = best_alt
    return preds

thresh_grid  = np.arange(0.10, 0.91, 0.05)
best_thresh  = np.full(NC, 0.5)

# Sequential coordinate search — each class optimised in turn
for cls_idx in range(NC):
    best_f1_here = -1.0
    best_t_here  = best_thresh[cls_idx]
    for t in thresh_grid:
        thresholds          = best_thresh.copy()
        thresholds[cls_idx] = t
        preds = apply_thresholds_vec(va_probs, thresholds)
        f1_t  = f1_score(va_true, preds, average='macro', zero_division=0)
        if f1_t > best_f1_here:
            best_f1_here = f1_t
            best_t_here  = t
    best_thresh[cls_idx] = best_t_here

# Print threshold table
print(f"\n{'Class':<8} {'Thresh':>8} {'Default F1':>12} {'Tuned F1':>10}")
print("-" * 42)
va_preds_cal = apply_thresholds_vec(va_probs, best_thresh)
for cls_idx, cls_name in enumerate(CLASSES):
    default_f1 = f1_score(va_true == cls_idx,
                          argmax_preds == cls_idx, zero_division=0)
    tuned_f1   = f1_score(va_true == cls_idx,
                          va_preds_cal == cls_idx,  zero_division=0)
    flag = " ←" if tuned_f1 > default_f1 + 0.005 else ""
    print(f"  {cls_name:<6} {best_thresh[cls_idx]:>8.2f} "
          f"{default_f1:>12.4f} {tuned_f1:>10.4f}{flag}")

final_val_f1 = f1_score(va_true, va_preds_cal, average='macro', zero_division=0)
final_val_ba = balanced_accuracy_score(va_true, va_preds_cal)
final_val_acc = accuracy_score(va_true, va_preds_cal)
print(f"\n  Val Macro-F1 after calibration : {final_val_f1:.4f}")
print(f"  Val Balanced-Acc               : {final_val_ba:.4f}")
print(f"  Val Accuracy                   : {final_val_acc:.4f}")

# Full classification report on val (safe — all labels 0..12)
print("\nClassification report on val_df (T-scaled + Calibrated):")
print(classification_report(
    va_true, va_preds_cal,
    target_names=CLASSES,
    labels=list(range(NC)),
    digits=4, zero_division=0
))

# ════════════════════════════════════════════════════════════════
# Step 4 — Test inference (NO metrics — test_df has no real labels)
# ════════════════════════════════════════════════════════════════
print("\n[3/4] Test inference (label-free) ...")

try:
    _ = test_loader
    print("  Using test_loader from Cell 12.")
except NameError:
    test_loader = DataLoader(
        WBCDataset(test_df, mode='test'),
        batch_size  = CFG['batch_size'],
        shuffle     = False,
        num_workers = CFG['num_workers'],
        pin_memory  = CFG['pin_memory'],
    )
    print("  test_loader rebuilt from test_df.")

te_logits_list = []
with torch.no_grad():
    for imgs, _ in test_loader:        # labels ignored completely
        imgs = imgs.to(device)
        with autocast():
            logits = final_model(imgs)
        te_logits_list.append(logits.cpu().float())

te_logits    = torch.cat(te_logits_list, dim=0)
te_probs     = F.softmax(te_logits / best_T, dim=1).numpy()
te_preds_raw = te_probs.argmax(axis=1)
te_preds_cal = apply_thresholds_vec(te_probs, best_thresh)

print(f"  {len(te_preds_cal):,} test predictions generated.")

# Prediction distribution
print("\n  Prediction distribution (calibrated):")
import pandas as pd
dist = pd.Series(te_preds_cal).map(IDX2CLASS).value_counts().sort_index()
for cls, cnt in dist.items():
    bar = "█" * int(cnt / dist.max() * 25)
    print(f"    {cls:<6} {cnt:>6,}  {bar}")

# ════════════════════════════════════════════════════════════════
# Step 5 — Save
# ════════════════════════════════════════════════════════════════
print("\n[4/4] Saving ...")

# Calibration params
calib_data = {
    'temperature' : best_T,
    'thresholds'  : best_thresh.tolist(),
    'classes'     : CLASSES,
    'val_macro_f1': float(final_val_f1),
    'val_bal_acc' : float(final_val_ba),
}
calib_path = os.path.join(CFG['save_dir'], 'calibration.pkl')
with open(calib_path, 'wb') as f:
    pickle.dump(calib_data, f)

# Test predictions CSV
test_pred_df = test_df.copy()
test_pred_df['predicted_label']     = [IDX2CLASS[p] for p in te_preds_cal]
test_pred_df['predicted_label_idx'] = te_preds_cal
for i, cls in enumerate(CLASSES):
    test_pred_df[f'prob_{cls}'] = te_probs[:, i]

pred_path = os.path.join(CFG['save_dir'], 'test_predictions_calibrated.csv')
test_pred_df.to_csv(pred_path, index=False)

print(f"\n  ✓ calibration.pkl                 → {calib_path}")
print(f"  ✓ test_predictions_calibrated.csv → {pred_path}")
print("\n" + "=" * 60)
print("  CALIBRATION COMPLETE")
print(f"  Temperature           : {best_T:.2f}")
print(f"  Val Macro-F1          : {final_val_f1:.4f}")
print(f"  Val Balanced-Acc      : {final_val_ba:.4f}")
print(f"  Test predictions      : {len(te_preds_cal):,}")
print("=" * 60)

In [ ]:
# ============================================================
# REBUILD final_te_loader
# NO RETRAINING
# ============================================================

print("=" * 60)
print("REBUILDING TEST LOADER")
print("=" * 60)

# test dataset
final_te_ds = WBCDataset(
    test_df,
    mode='test'
)

# test loader
final_te_loader = DataLoader(
    final_te_ds,
    batch_size = CFG['batch_size'],
    shuffle    = False,
    num_workers= CFG['num_workers'],
    pin_memory = CFG['pin_memory']
)

print("final_te_loader rebuilt successfully.")
print(f"Total test batches: {len(final_te_loader)}")

In [ ]:
# ============================================================
# REBUILD te_preds + te_targets
# ============================================================

print("=" * 60)
print("REBUILDING PREDICTIONS")
print("=" * 60)

final_model.eval()

te_preds   = []
te_targets = []

with torch.no_grad():

    for imgs, labels in final_te_loader:

        imgs = imgs.to(device)

        outputs = final_model(imgs)

        preds = outputs.argmax(dim=1).cpu().numpy()

        te_preds.extend(preds)

        te_targets.extend(labels.numpy())

print(f"Predictions rebuilt : {len(te_preds)}")
print(f"Targets rebuilt     : {len(te_targets)}")

print("\nDone.")

In [ ]:
# ============================================================
# STEP 1 — REBUILD TEST LOADER
# ============================================================

print("=" * 60)
print("REBUILDING TEST LOADER")
print("=" * 60)

final_te_ds = WBCDataset(
    test_df,
    mode='test'
)

final_te_loader = DataLoader(
    final_te_ds,
    batch_size = CFG['batch_size'],
    shuffle    = False,
    num_workers= CFG['num_workers'],
    pin_memory = CFG['pin_memory']
)

print("SUCCESS:")
print(f"Total batches = {len(final_te_loader)}")

In [ ]:
# ============================================================
# STEP 2 — REBUILD te_preds + te_targets
# ============================================================

print("=" * 60)
print("REBUILDING PREDICTIONS")
print("=" * 60)

final_model.eval()

te_preds   = []
te_targets = []

with torch.no_grad():

    for imgs, labels in final_te_loader:

        imgs = imgs.to(device)

        outputs = final_model(imgs)

        preds = outputs.argmax(dim=1).cpu().numpy()

        te_preds.extend(preds)

        te_targets.extend(labels.numpy())

print("\nSUCCESS")
print(f"Predictions : {len(te_preds)}")
print(f"Targets     : {len(te_targets)}")

In [ ]:
# ===============================================================
# FINAL CORRECTED GRADCAM VALIDATION CELL
# 13-CLASS WBCBENCH2026
# FULL FIXED VERSION
# ===============================================================

import os
import cv2
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ===============================================================
# GRADCAM ENGINE
# ===============================================================

class GradCAM:

    def __init__(self, model):

        self.model       = model
        self.gradients   = None
        self.activations = None
        self.hooks       = []

        target_layer = model.stages[-1]

        self.hooks.append(
            target_layer.register_forward_hook(
                self.save_activation
            )
        )

        self.hooks.append(
            target_layer.register_full_backward_hook(
                self.save_gradient
            )
        )

    def save_activation(self, module, inp, out):

        self.activations = out.detach()

    def save_gradient(self, module, grad_in, grad_out):

        self.gradients = grad_out[0].detach()

    def __call__(self, x, class_idx=None):

        self.model.eval()

        x = x.requires_grad_(True)

        logits = self.model(x)

        pred_idx = logits.argmax(dim=1).item()

        if class_idx is None:
            class_idx = pred_idx

        self.model.zero_grad()

        logits[0, class_idx].backward()

        weights = self.gradients.mean(
            dim=[2, 3],
            keepdim=True
        )

        cam = (
            weights * self.activations
        ).sum(dim=1, keepdim=True)

        cam = F.relu(cam)

        cam = F.interpolate(
            cam,
            size=(224, 224),
            mode='bilinear',
            align_corners=False
        )

        cam = cam.squeeze().cpu().numpy()

        if cam.max() > cam.min():

            cam = (
                cam - cam.min()
            ) / (
                cam.max() - cam.min()
            )

        return cam, pred_idx

    def remove_hooks(self):

        for h in self.hooks:
            h.remove()

# ===============================================================
# HEATMAP OVERLAY
# ===============================================================

def overlay_heatmap(img_rgb, heatmap, alpha=0.45):

    heatmap = (heatmap * 255).astype(np.uint8)

    jet = cv2.applyColorMap(
        heatmap,
        cv2.COLORMAP_JET
    )

    jet = cv2.cvtColor(
        jet,
        cv2.COLOR_BGR2RGB
    )

    out = (
        alpha * jet + (1 - alpha) * img_rgb
    ).astype(np.uint8)

    return out

# ===============================================================
# PICK ONE SAMPLE PER CLASS
# ===============================================================

def pick_one_per_class(df, classes):

    rows = []

    label_col = (
        'label'
        if 'label' in df.columns
        else 'labels'
    )

    for cls in classes:

        sub = df[
            df[label_col] == cls
        ]

        if len(sub) > 0:

            rows.append(
                sub.sample(
                    1,
                    random_state=42
                )
            )

    return pd.concat(rows).reset_index(drop=True)

# ===============================================================
# BEFORE VS AFTER ROI
# ===============================================================

def plot_before_after_roi_gradcam(
    df,
    model,
    save_dir,
    n_classes=13
):

    print("=" * 60)
    print("Generating BEFORE vs AFTER ROI GradCAM")
    print("=" * 60)

    CLASSES = CFG['classes'][:n_classes]

    label_col = (
        'label'
        if 'label' in df.columns
        else 'labels'
    )

    gcam = GradCAM(model)

    sample_df = pick_one_per_class(
        df,
        CLASSES
    )

    n_rows = len(sample_df)

    fig = plt.figure(
        figsize=(16, 3.5 * n_rows)
    )

    gs = gridspec.GridSpec(
        n_rows,
        4,
        figure=fig,
        hspace=0.20,
        wspace=0.05
    )

    titles = [
        'Original',
        'GradCAM Original',
        'ROI Image',
        'GradCAM ROI'
    ]

    for row_i, (_, row) in enumerate(sample_df.iterrows()):

        img_bgr = cv2.imread(row['path'])

        if img_bgr is None:
            continue

        img_rgb = cv2.cvtColor(
            img_bgr,
            cv2.COLOR_BGR2RGB
        )

        img_rgb = cv2.resize(
            img_rgb,
            (224, 224)
        )

        tensor = val_tf(img_rgb).unsqueeze(0).to(device)

        with torch.enable_grad():

            cam, pred_idx = gcam(
                tensor,
                class_idx=int(row['label_idx'])
            )

        overlay = overlay_heatmap(
            img_rgb,
            cam
        )

        imgs = [
            img_rgb,
            overlay,
            img_rgb,
            overlay
        ]

        for col_i, im in enumerate(imgs):

            ax = fig.add_subplot(
                gs[row_i, col_i]
            )

            ax.imshow(im)

            ax.set_xticks([])
            ax.set_yticks([])

            if row_i == 0:

                ax.set_title(
                    titles[col_i],
                    fontsize=8,
                    fontweight='bold'
                )

            if col_i == 0:

                true_lbl = row[label_col]

                ax.set_ylabel(
                    true_lbl,
                    fontsize=8,
                    rotation=0,
                    labelpad=45
                )

            if col_i in [1, 3]:

                pred_lbl = IDX2CLASS[pred_idx]

                color = (
                    'lime'
                    if pred_lbl == row[label_col]
                    else 'red'
                )

                ax.text(
                    0.03,
                    0.03,
                    f"Pred:{pred_lbl}",
                    transform=ax.transAxes,
                    fontsize=7,
                    color=color,
                    bbox=dict(
                        facecolor='black',
                        alpha=0.6
                    )
                )

    out_path = os.path.join(
        save_dir,
        'gradcam_before_after_roi.png'
    )

    plt.tight_layout()

    plt.savefig(
        out_path,
        dpi=130,
        bbox_inches='tight'
    )

    plt.show()

    plt.close()

    print(f"Saved → {out_path}")

    gcam.remove_hooks()

# ===============================================================
# CORRECT / WRONG PREDICTIONS
# ===============================================================

def plot_gradcam_predictions(
    df,
    model,
    preds,
    targets,
    save_dir
):

    print("=" * 60)
    print("Generating Correct/Wrong GradCAM")
    print("=" * 60)

    gcam = GradCAM(model)

    preds   = np.array(preds)
    targets = np.array(targets)

    correct_idx = np.where(
        preds == targets
    )[0][:13]

    wrong_idx = np.where(
        preds != targets
    )[0][:10]

    def make_panel(indices, title, fname):

        if len(indices) == 0:

            print(f"Skipping {title} — no samples.")
            return

        cols = 5
        rows = int(np.ceil(len(indices) / cols))

        fig, axes = plt.subplots(
            rows,
            cols,
            figsize=(18, 4 * rows)
        )

        axes = np.array(axes).reshape(-1)

        fig.suptitle(
            title,
            fontsize=12,
            fontweight='bold'
        )

        for ax in axes:
            ax.axis('off')

        for i, idx in enumerate(indices):

            row = df.iloc[idx]

            img_bgr = cv2.imread(row['path'])

            if img_bgr is None:
                continue

            img_rgb = cv2.cvtColor(
                img_bgr,
                cv2.COLOR_BGR2RGB
            )

            img_rgb = cv2.resize(
                img_rgb,
                (224, 224)
            )

            tensor = val_tf(img_rgb).unsqueeze(0).to(device)

            with torch.enable_grad():

                cam, pred_idx = gcam(
                    tensor,
                    class_idx=int(row['label_idx'])
                )

            overlay = overlay_heatmap(
                img_rgb,
                cam
            )

            axes[i].imshow(overlay)

            true_lbl = IDX2CLASS[
                int(row['label_idx'])
            ]

            pred_lbl = IDX2CLASS[pred_idx]

            color = (
                'lime'
                if true_lbl == pred_lbl
                else 'red'
            )

            axes[i].set_title(
                f"T:{true_lbl}\nP:{pred_lbl}",
                fontsize=8,
                color=color
            )

        plt.tight_layout()

        out_path = os.path.join(
            save_dir,
            fname
        )

        plt.savefig(
            out_path,
            dpi=120,
            bbox_inches='tight'
        )

        plt.show()

        plt.close()

        print(f"Saved → {out_path}")

    make_panel(
        correct_idx,
        'GradCAM Correct Predictions',
        'gradcam_correct_predictions.png'
    )

    make_panel(
        wrong_idx,
        'GradCAM Wrong Predictions',
        'gradcam_wrong_predictions.png'
    )

    gcam.remove_hooks()

# ===============================================================
# BUILD VALIDATION PREDICTIONS
# ===============================================================

print("=" * 60)
print("BUILDING VALIDATION PREDICTIONS")
print("=" * 60)

val_loader = DataLoader(
    WBCDataset(val_df, mode='val'),
    batch_size = CFG['batch_size'],
    shuffle    = False,
    num_workers= CFG['num_workers'],
    pin_memory = CFG['pin_memory']
)

final_model.eval()

va_preds   = []
va_targets = []

with torch.no_grad():

    for imgs, labels in val_loader:

        imgs = imgs.to(device)

        outputs = final_model(imgs)

        preds = outputs.argmax(dim=1).cpu().numpy()

        va_preds.extend(preds)

        va_targets.extend(labels.numpy())

print(f"Predictions : {len(va_preds)}")
print(f"Targets     : {len(va_targets)}")

# ===============================================================
# RUN GRADCAM
# ===============================================================

print("=" * 60)
print("FINAL GRADCAM VALIDATION")
print("=" * 60)

plot_before_after_roi_gradcam(
    val_df,
    final_model,
    CFG['save_dir'],
    n_classes=13
)

plot_gradcam_predictions(
    val_df,
    final_model,
    va_preds,
    va_targets,
    CFG['save_dir']
)

print("\nGradCAM validation complete.")
print("All GradCAM files saved successfully.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL B — Error Analysis with Biological Interpretation
#
# WHERE TO ADD:
#   AFTER  Cell A (GradCAM above)
#   BEFORE Cell 37 (training curves visualization)
#
# WHAT THIS PRODUCES:
#   1. Full confusion matrix (13×13, counts + normalised)
#   2. Confused pairs table with biological reason
#   3. Per-class precision / recall / F1 bar chart
#   4. Hardest classes ranked by F1
#   All saved as PNG to CFG['save_dir']
# ══════════════════════════════════════════════════════════════════
 
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, f1_score,
    precision_score, recall_score
)
 
print("=" * 60)
print("ERROR ANALYSIS — Biological Interpretation")
print("=" * 60)
 
CLASSES = CFG['classes']
NC      = CFG['num_classes']
 
# ── Biological confusion explanation table ────────────────────────
# Each pair: (true_class, pred_class): biological reason
BIO_CONFUSION = {
    ('MY',  'MMY') : 'Maturation continuity — nuclear indentation gradual',
    ('MMY', 'MY')  : 'Maturation continuity — nuclear indentation gradual',
    ('PMY', 'MY')  : 'Precursor similarity — both have large nuclei + azurophilic granules',
    ('MY',  'PMY') : 'Precursor similarity — both have large nuclei + azurophilic granules',
    ('LY',  'VLY') : 'Reactive morphology — VLY is activated LY with similar nucleus',
    ('VLY', 'LY')  : 'Reactive morphology — VLY is activated LY with similar nucleus',
    ('SNE', 'BNE') : 'Lobe overlap — BNE nucleus not yet segmented, looks like early SNE',
    ('BNE', 'SNE') : 'Lobe overlap — SNE multi-lobe vs BNE band shape can be ambiguous',
    ('EO',  'BA')  : 'Granule similarity — both have large granules, differ by colour',
    ('BA',  'EO')  : 'Granule similarity — basophilic vs eosinophilic granule stain varies',
    ('BL',  'MY')  : 'Blast-myelocyte overlap — BL has fine chromatin, MY has coarser',
    ('MY',  'BL')  : 'Blast-myelocyte overlap — BL has fine chromatin, MY has coarser',
    ('PLY', 'LY')  : 'Prolymphocyte vs lymphocyte — PLY has prominent nucleolus',
    ('LY',  'PLY') : 'Prolymphocyte vs lymphocyte — PLY has prominent nucleolus',
    ('PC',  'LY')  : 'Plasma cell vs lymphocyte — PC clock-face chromatin sometimes subtle',
    ('LY',  'PC')  : 'Plasma cell vs lymphocyte — PC clock-face chromatin sometimes subtle',
    ('PMY', 'BL')  : 'Promyelocyte vs blast — both are large immature cells',
    ('BL',  'PMY') : 'Promyelocyte vs blast — both are large immature cells',
}
 
 
def plot_error_analysis(te_preds, te_targets, save_dir):
    te_preds_arr   = np.array(te_preds)
    te_targets_arr = np.array(te_targets)
 
    # ── 1. Confusion matrix ───────────────────────────────────────
    cm      = confusion_matrix(te_targets_arr, te_preds_arr,
                               labels=list(range(NC)))
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
 
    fig, axes = plt.subplots(1, 2, figsize=(22, 9))
    fig.suptitle(
        'Test Set — Confusion Matrix (13-class)\n'
        'ConvNeXt-Tiny + HybridLoss + ROI Preprocessing',
        fontsize=12, fontweight='bold'
    )
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES,
                ax=axes[0], linewidths=0.3,
                annot_kws={'size': 7})
    axes[0].set_title('Raw Counts', fontsize=11)
    axes[0].set_xlabel('Predicted', fontsize=10)
    axes[0].set_ylabel('True',      fontsize=10)
    axes[0].tick_params(axis='x', rotation=45, labelsize=8)
    axes[0].tick_params(axis='y', rotation=0,  labelsize=8)
 
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES,
                ax=axes[1], linewidths=0.3,
                vmin=0, vmax=1, annot_kws={'size': 7})
    axes[1].set_title('Normalised (row = true class)', fontsize=11)
    axes[1].set_xlabel('Predicted', fontsize=10)
    axes[1].set_ylabel('True',      fontsize=10)
    axes[1].tick_params(axis='x', rotation=45, labelsize=8)
    axes[1].tick_params(axis='y', rotation=0,  labelsize=8)
 
    plt.tight_layout()
    out = os.path.join(save_dir, 'error_analysis_confusion_matrix.png')
    plt.savefig(out, dpi=120, bbox_inches='tight', facecolor='white')
    plt.show(); plt.close(fig)
    print(f"  Saved → {out}")
 
    # ── 2. Top confused pairs + biological reason ─────────────────
    print("\n" + "=" * 72)
    print("  TOP CONFUSED PAIRS — BIOLOGICAL INTERPRETATION")
    print("=" * 72)
    print(f"  {'True':<6} {'Pred':<6} {'Count':>6}  Biological Reason")
    print("-" * 72)
 
    # Extract off-diagonal confusion counts
    pairs = []
    for i in range(NC):
        for j in range(NC):
            if i != j and cm[i, j] > 0:
                pairs.append((cm[i, j], CLASSES[i], CLASSES[j]))
    pairs.sort(reverse=True)
 
    for count, true_cls, pred_cls in pairs[:15]:
        reason = BIO_CONFUSION.get(
            (true_cls, pred_cls),
            'Morphological overlap — review stain quality'
        )
        print(f"  {true_cls:<6} {pred_cls:<6} {count:>6}  {reason}")
    print("=" * 72)
 
    # ── 3. Per-class P/R/F1 bar chart ────────────────────────────
    per_p  = precision_score(te_targets_arr, te_preds_arr,
                             average=None, zero_division=0,
                             labels=list(range(NC)))
    per_r  = recall_score(te_targets_arr, te_preds_arr,
                          average=None, zero_division=0,
                          labels=list(range(NC)))
    per_f1 = f1_score(te_targets_arr, te_preds_arr,
                      average=None, zero_division=0,
                      labels=list(range(NC)))
 
    x     = np.arange(NC)
    width = 0.26
    fig, ax = plt.subplots(figsize=(16, 5))
    ax.bar(x - width, per_p,  width, label='Precision', color='steelblue',  alpha=0.85)
    ax.bar(x,         per_r,  width, label='Recall',    color='darkorange', alpha=0.85)
    ax.bar(x + width, per_f1, width, label='F1',        color='seagreen',   alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(CLASSES, fontsize=9)
    ax.set_ylim(0, 1.08)
    ax.set_ylabel('Score', fontsize=10)
    ax.set_title(
        'Per-Class Precision / Recall / F1 — Test Set\n'
        'Classes sorted by CFG order (SNE→PLY)',
        fontsize=11, fontweight='bold'
    )
    ax.legend(fontsize=9)
    ax.axhline(0.5, color='red', linestyle='--', linewidth=0.8, alpha=0.5,
               label='0.5 threshold')
    ax.grid(axis='y', alpha=0.3)
 
    # Annotate F1 values on bars
    for i, f in enumerate(per_f1):
        ax.text(i + width, f + 0.015, f'{f:.2f}',
                ha='center', va='bottom', fontsize=6.5, color='black')
 
    plt.tight_layout()
    out = os.path.join(save_dir, 'error_analysis_per_class_metrics.png')
    plt.savefig(out, dpi=120, bbox_inches='tight', facecolor='white')
    plt.show(); plt.close(fig)
    print(f"  Saved → {out}")
 
    # ── 4. Hardest classes ranked ─────────────────────────────────
    ranked = sorted(zip(CLASSES, per_f1), key=lambda x: x[1])
    print("\n  HARDEST CLASSES (lowest F1 on test set):")
    print(f"  {'Rank':<6} {'Class':<8} {'F1':>6}  Likely Confusion")
    print("-" * 52)
    for rank, (cls, f) in enumerate(ranked, 1):
        # Find most common mistake for this class
        cls_idx   = CLASS2IDX[cls]
        row_cm    = cm[cls_idx].copy()
        row_cm[cls_idx] = 0
        if row_cm.sum() > 0:
            worst_pred = CLASSES[row_cm.argmax()]
            reason     = BIO_CONFUSION.get(
                (cls, worst_pred), 'See confusion matrix'
            )
        else:
            worst_pred = 'none'
            reason     = 'Classified correctly in all cases'
        print(f"  {rank:<6} {cls:<8} {f:>6.3f}  "
              f"most confused with {worst_pred}: {reason[:45]}")
    print("=" * 52)
 
    # ── 5. Summary print ─────────────────────────────────────────
    macro_f1 = f1_score(te_targets_arr, te_preds_arr,
                        average='macro', zero_division=0)
    wt_f1    = f1_score(te_targets_arr, te_preds_arr,
                        average='weighted', zero_division=0)
    print(f"\n  Macro F1   : {macro_f1:.4f}")
    print(f"  Weighted F1: {wt_f1:.4f}")
    print(f"\n  Mean F1 easy classes  (SNE,LY,MO,EO): "
          f"{np.mean([per_f1[CLASS2IDX[c]] for c in ['SNE','LY','MO','EO']]):.3f}")
    print(f"  Mean F1 rare  classes (BA,PLY,PC,VLY): "
          f"{np.mean([per_f1[CLASS2IDX[c]] for c in ['BA','PLY','PC','VLY']]):.3f}")
    print(f"  Mean F1 immature myeloid (MY,MMY,PMY) : "
          f"{np.mean([per_f1[CLASS2IDX[c]] for c in ['MY','MMY','PMY']]):.3f}")
 
 
# RUN ERROR ANALYSIS
# ============================================================
# BUILD VALIDATION PREDICTIONS
# ============================================================

print("=" * 60)
print("BUILDING VALIDATION PREDICTIONS")
print("=" * 60)

val_loader = DataLoader(
    WBCDataset(val_df, mode='val'),
    batch_size = CFG['batch_size'],
    shuffle    = False,
    num_workers= CFG['num_workers'],
    pin_memory = CFG['pin_memory']
)

final_model.eval()

va_preds   = []
va_targets = []

with torch.no_grad():

    for imgs, labels in val_loader:

        imgs = imgs.to(device)

        outputs = final_model(imgs)

        preds = outputs.argmax(dim=1).cpu().numpy()

        va_preds.extend(preds)

        va_targets.extend(labels.numpy())

print(f"Validation predictions : {len(va_preds)}")
print(f"Validation targets     : {len(va_targets)}")

# ============================================================
# RUN ERROR ANALYSIS
# ============================================================

plot_error_analysis(
    va_preds,
    va_targets,
    save_dir=CFG['save_dir']
)

print("\nError analysis complete.")
print("Files saved:")
print(f"  {CFG['save_dir']}/error_analysis_confusion_matrix.png")
print(f"  {CFG['save_dir']}/error_analysis_per_class_metrics.png")

## Cell 13 — Visualizations

In [ ]:
CLASSES = CFG['classes']
os.makedirs(CFG['save_dir'], exist_ok=True)

# ── 1. Loss & accuracy curves (all folds + final) ──────────────────────────
n_folds = len(fold_results)
fig, axes = plt.subplots(2, n_folds+1, figsize=(5*(n_folds+1), 8))
fig.suptitle('Training Curves — ConvNeXt-Tiny (WBC 13-class)',
             fontsize=14, fontweight='bold')

for i, r in enumerate(fold_results):
    h = r['history']
    ep = range(1, len(h['tr_loss'])+1)
    axes[0,i].plot(ep, h['tr_loss'], label='Train', color='steelblue')
    axes[0,i].plot(ep, h['va_loss'], label='Val',   color='darkorange')
    axes[0,i].set_title(f"Fold {r['fold']} — Loss"); axes[0,i].legend()
    axes[0,i].set_xlabel('Epoch'); axes[0,i].set_ylabel('Loss')

    axes[1,i].plot(ep, h['tr_acc'], label='Train', color='steelblue')
    axes[1,i].plot(ep, h['va_acc'], label='Val',   color='darkorange')
    axes[1,i].plot(ep, h['va_f1'],  label='Val F1',color='green', linestyle='--')
    axes[1,i].set_title(f"Fold {r['fold']} — Acc/F1"); axes[1,i].legend()
    axes[1,i].set_xlabel('Epoch'); axes[1,i].set_ylabel('Score')

# Final model curves
h = final_hist; ep = range(1, len(h['tr_loss'])+1)
axes[0,-1].plot(ep, h['tr_loss'], color='steelblue', label='Train')
axes[0,-1].plot(ep, h['va_loss'], color='darkorange', label='Val')
axes[0,-1].set_title('Final Model — Loss'); axes[0,-1].legend()
axes[1,-1].plot(ep, h['tr_acc'], color='steelblue', label='Train')
axes[1,-1].plot(ep, h['va_acc'], color='darkorange', label='Val')
axes[1,-1].plot(ep, h['va_f1'],  color='green', linestyle='--', label='Val F1')
axes[1,-1].set_title('Final Model — Acc/F1'); axes[1,-1].legend()

plt.tight_layout()
plt.savefig(f"{CFG['save_dir']}/training_curves.png", dpi=120, bbox_inches='tight')
plt.show()
print("Saved: training_curves.png")
# ============================================================
# VALIDATION METRICS
# ============================================================

from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    accuracy_score
)

va_f1  = f1_score(
    va_targets,
    va_preds,
    average='macro',
    zero_division=0
)

va_acc = accuracy_score(
    va_targets,
    va_preds
)

va_ba = balanced_accuracy_score(
    va_targets,
    va_preds
)

# ============================================================
# OOF + VALIDATION CONFUSION MATRICES
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)

for ax, (preds, targets, title) in zip(axes, [

    (
        oof_preds,
        oof_targets,
        f'OOF Confusion Matrix\nmacro-F1={oof_f1:.4f}'
    ),

    (
        va_preds,
        va_targets,
        f'Validation Confusion Matrix\nmacro-F1={va_f1:.4f}'
    ),

]):

    cm = confusion_matrix(
        targets,
        preds
    )

    cm_norm = cm.astype(float) / (
        cm.sum(axis=1, keepdims=True) + 1e-9
    )

    annot = np.array([
        [
            f'{cm[i,j]}\n({cm_norm[i,j]*100:.1f}%)'
            for j in range(len(CLASSES))
        ]
        for i in range(len(CLASSES))
    ])

    sns.heatmap(
        cm_norm,
        annot=annot,
        fmt='s',
        cmap='Blues',
        xticklabels=CLASSES,
        yticklabels=CLASSES,
        linewidths=0.5,
        ax=ax,
        vmin=0,
        vmax=1
    )

    ax.set_title(
        title,
        fontsize=12,
        fontweight='bold'
    )

    ax.set_xlabel('Predicted')

    ax.set_ylabel('True')

plt.tight_layout()

plt.savefig(
    f"{CFG['save_dir']}/confusion_matrices.png",
    dpi=120,
    bbox_inches='tight'
)

plt.show()

print("Saved: confusion_matrices.png")

# ============================================================
# PER-CLASS F1
# ============================================================

from sklearn.metrics import f1_score as f1s

fig, ax = plt.subplots(
    figsize=(16,5)
)

oof_per = f1s(
    oof_targets,
    oof_preds,
    average=None,
    zero_division=0
)

va_per = f1s(
    va_targets,
    va_preds,
    average=None,
    zero_division=0
)

x = np.arange(len(CLASSES))

w = 0.35

bars1 = ax.bar(
    x-w/2,
    oof_per,
    w,
    label='OOF',
    color='steelblue',
    alpha=0.85
)

bars2 = ax.bar(
    x+w/2,
    va_per,
    w,
    label='Validation',
    color='darkorange',
    alpha=0.85
)

for bar in list(bars1) + list(bars2):

    ax.text(
        bar.get_x()+bar.get_width()/2,
        bar.get_height()+0.01,
        f'{bar.get_height():.3f}',
        ha='center',
        va='bottom',
        fontsize=9
    )

ax.set_xticks(x)

ax.set_xticklabels(
    CLASSES,
    fontsize=11
)

ax.set_ylim(0,1.1)

ax.set_ylabel('F1 Score')

ax.set_xlabel('Class')

ax.set_title(
    'Per-class F1 — OOF vs Validation',
    fontsize=13,
    fontweight='bold'
)

ax.legend()

ax.grid(axis='y', alpha=0.3)

plt.tight_layout()

plt.savefig(
    f"{CFG['save_dir']}/per_class_f1.png",
    dpi=120,
    bbox_inches='tight'
)

plt.show()

print("Saved: per_class_f1.png")

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "="*60)

print("FINAL RESULTS SUMMARY — ConvNeXt-Tiny (13-class WBC)")

print("="*60)

print(f"{'Metric':<30} {'OOF':>8} {'VAL':>8}")

print("-"*60)

print(
    f"{'Accuracy':<30} "
    f"{accuracy_score(oof_targets,oof_preds):>8.4f} "
    f"{va_acc:>8.4f}"
)

print(
    f"{'Macro F1':<30} "
    f"{oof_f1:>8.4f} "
    f"{va_f1:>8.4f}"
)

print(
    f"{'Balanced Accuracy':<30} "
    f"{oof_ba:>8.4f} "
    f"{va_ba:>8.4f}"
)

for i,c in enumerate(CLASSES):

    print(
        f"{'F1 — '+c:<30} "
        f"{oof_per[i]:>8.4f} "
        f"{va_per[i]:>8.4f}"
    )

print("="*60)

print(f"\nAll plots saved to: {CFG['save_dir']}/")



## Cell 14 — Save final model & results to Drive

In [ ]:

import pickle

# Save final model weights
torch.save(
    final_model.state_dict(),
    f"{CFG['save_dir']}/convnext_final.pth"
)

# Check what keys fold_results actually has
print("fold_results keys:", fold_results[0].keys() if fold_results else "empty")

# Build results dict safely
results = {
    'model'    : 'ConvNeXt-Tiny',
    'test_f1'  : float(te_f1),
    'test_acc' : float(te_acc),
    'test_ba'  : float(balanced_accuracy_score(te_targets, te_preds)),
    'oof_f1'   : float(oof_f1),
    'oof_ba'   : float(oof_ba),
    'oof_per_class'  : {c: float(oof_per[i]) for i, c in enumerate(CLASSES)},
    'test_per_class' : {c: float(te_per[i])  for i, c in enumerate(CLASSES)},
    'fold_results'   : fold_results,   # save as-is, no key extraction
    'mean_cv_f1'     : float(mean_f1),
    'mean_cv_acc'    : float(mean_acc),
}

with open(f"{CFG['save_dir']}/convnext_results.pkl", 'wb') as f:
    pickle.dump(results, f)

print("Saved to Drive:")
for fname in ['convnext_final.pth', 'convnext_results.pkl',
              'training_curves.png', 'confusion_matrices.png',
              'per_class_f1.png']:
    fpath = f"{CFG['save_dir']}/{fname}"
    if os.path.exists(fpath):
        kb = os.path.getsize(fpath) / 1024
        print(f"  ✓ {fname}  ({kb:.0f} KB)")
    else:
        print(f"  ✗ {fname}  (NOT FOUND)")

print("\nAll done! 13-class WBC pipeline complete.")
print(f"  Classes : {CFG['classes']}")
print(f"  Test F1 : {float(te_f1):.4f}")
print(f"  OOF F1  : {float(oof_f1):.4f}")
print("Session can now disconnect safely.")